# QIMED — Heisenberg com nested temporal, PCA e benchmarks clássicos

Este notebook é derivado de `QIMED_testes-quantum_nested_shap.ipynb`, sem modificar o original.

O notebook está dividido em duas partes independentes:

- **Parte I — 1.000 amostras:** benchmark clássico com todas as features, PCA-12, SHAP-12 e SHAP-36; depois Heisenberg 13q com PCA-12.
- **Parte II — 2.000 amostras:** benchmark clássico com todas as features, PCA-12, PCA-36, SHAP-12 e SHAP-36; depois Heisenberg 19q com PCA-36 e Heisenberg 37q com PCA-36/MPS.

Em todos os experimentos, preprocessing, PCA/SHAP e classificador são ajustados apenas no treino do respectivo split temporal. Nas camadas quânticas, PCA é usado somente para produzir a entrada da PQFM: o cenário clássico usa todas as features preprocessadas, o quantum-only usa todas as features quânticas e o híbrido concatena todas as clássicas com todas as quânticas. As camadas Heisenberg usam um circuito congelado por arquitetura e caches quânticos específicos das representações leakage-safe de cada fold.


In [1]:
import json
import pickle
import re
import time
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import shap
from scipy.stats import t as student_t
from scipy.stats import wilcoxon

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pqfmlib import HeisenbergProjectiveQFM

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
OUTER_SPLITS = 10
INNER_SPLITS = 3
SCORING = "average_precision"
SHOTS = 4096
METRICS = (
    "ROC-AUC", "PR-AUC", "Accuracy", "Precision",
    "Recall", "F1", "Specificity",
)

OUTPUT_ROOT = Path("outputs")
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


/home/rafael/projects/pqfmlib-api/.venv_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset e colunas

O dataset é carregado uma única vez. Cada parte cria sua própria amostra estratificada e depois restaura a ordem temporal por `period_start`.


In [2]:
DATA_PATH = Path("./data/dataset_modelo_readmissao.parquet")
TARGET_COL = "readmitted_30d"

FEATURE_COLS = [
    "age_at_enc", "gender", "race", "deceased", "marital_status",
    "class_code", "enc_type_grp", "has_reason",
    "n_enc_total", "n_enc_30d", "n_enc_90d", "n_enc_365d",
    "days_since_last", "had_emer_90d", "had_imp_90d",
    "month", "quarter", "day_of_week", "is_weekend",
    "has_renal_disease", "has_diabetes", "has_hypertension", "has_mental_health",
    "n_conditions_total", "n_conditions_active",
    "last_hba1c", "last_egfr", "last_systolic_bp", "last_diastolic_bp", "last_bmi",
    "n_labs_90d", "n_vitals_90d",
    "n_procedures_90d", "n_procedures_365d", "had_surgical_90d", "had_dialysis_90d",
]

NUM_COLS = [
    "age_at_enc", "n_enc_total", "n_enc_30d", "n_enc_90d", "n_enc_365d",
    "days_since_last", "n_conditions_total", "n_conditions_active",
    "n_procedures_90d", "n_procedures_365d", "n_labs_90d", "n_vitals_90d",
]
LAB_COLS = [
    "last_hba1c", "last_egfr", "last_systolic_bp", "last_diastolic_bp", "last_bmi",
]
CAT_COLS = [
    "gender", "race", "marital_status", "class_code", "enc_type_grp",
    "month", "day_of_week", "quarter",
]
BIN_COLS = [
    "deceased", "has_reason", "had_emer_90d", "had_imp_90d",
    "has_renal_disease", "has_diabetes", "has_hypertension", "has_mental_health",
    "had_surgical_90d", "had_dialysis_90d", "is_weekend",
]

df = pd.read_parquet(DATA_PATH)
print(f"Dataset completo: {df.shape}")
print(f"Distribuição: {df[TARGET_COL].value_counts(normalize=True).sort_index().round(4).to_dict()}")


Dataset completo: (7746, 43)
Distribuição: {0: 0.3195, 1: 0.6805}


## 2. Esquema metodológico

```text
TimeSeriesSplit externo: 10 folds
│
├── outer train
│   └── TimeSeriesSplit interno: 3 folds
│       ├── fit preprocessing no inner train
│       ├── fit PCA e/ou SHAP no inner train
│       ├── transform do inner validation
│       └── tuning por Average Precision
│
└── outer refit
    ├── refit preprocessing + PCA/SHAP no outer train completo
    ├── transform do outer test
    └── métricas finais do fold
```

A Heisenberg não usa MI. Seu circuito é preparado uma única vez por arquitetura. Contudo, `transform()` é aplicado separadamente às representações PCA específicas de cada fold, pois o preprocessing e a PCA continuam sendo ajustados apenas no treino.

Nos experimentos Heisenberg, PCA serve exclusivamente como entrada do mapa quântico. A comparação usa sempre: clássico = todas as features preprocessadas; quantum-only = todas as features quânticas; híbrido = todas as clássicas + todas as quânticas.


## 3. Funções comuns: amostragem, preprocessing, PCA e SHAP


In [3]:
def stratified_temporal_sample(
    frame,
    n_samples,
    target_col=TARGET_COL,
    random_state=RANDOM_STATE,
):
    if len(frame) < n_samples:
        raise ValueError(
            f"O dataset possui {len(frame)} linhas; são necessárias pelo menos {n_samples}."
        )
    ordered = (
        frame.sort_values("period_start", kind="mergesort")
        .reset_index()
        .rename(columns={"index": "source_index"})
    )
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=n_samples,
        random_state=random_state,
    )
    sample_idx, _ = next(splitter.split(ordered, ordered[target_col]))
    return (
        ordered.iloc[sample_idx]
        .sort_values("period_start", kind="mergesort")
        .reset_index(drop=True)
    )


def make_outer_folds(X, n_splits=OUTER_SPLITS):
    splitter = TimeSeriesSplit(n_splits=n_splits)
    return [(train.copy(), test.copy()) for train, test in splitter.split(X)]


def describe_nested_sizes(outer_folds):
    rows = []
    for outer_fold, (train_idx, test_idx) in enumerate(outer_folds, 1):
        inner_sizes = [
            len(inner_train)
            for inner_train, _ in TimeSeriesSplit(n_splits=INNER_SPLITS).split(train_idx)
        ]
        rows.append({
            "outer_fold": outer_fold,
            "outer_train": len(train_idx),
            "outer_test": len(test_idx),
            "inner_train_sizes": inner_sizes,
        })
    return pd.DataFrame(rows)


def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUM_COLS),
            ("labs", Pipeline([
                ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                ("scaler", StandardScaler()),
            ]), LAB_COLS),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), CAT_COLS),
            ("bin", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ]), BIN_COLS),
        ]
    )


def _as_dense_frame(values, columns):
    if hasattr(values, "toarray"):
        values = values.toarray()
    return pd.DataFrame(
        np.asarray(values, dtype=float),
        columns=list(columns),
    ).reset_index(drop=True)


def fit_preprocessing(X_train):
    preprocessor = build_preprocessor()
    transformed = preprocessor.fit_transform(X_train)
    columns = list(preprocessor.get_feature_names_out())
    X_frame = _as_dense_frame(transformed, columns)

    final_scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        final_scaler.fit_transform(X_frame),
        columns=columns,
    )
    fitted = {
        "preprocessor": preprocessor,
        "final_scaler": final_scaler,
        "columns": columns,
    }
    return fitted, X_scaled


def transform_preprocessing(fitted, X):
    transformed = fitted["preprocessor"].transform(X)
    X_frame = _as_dense_frame(transformed, fitted["columns"])
    return pd.DataFrame(
        fitted["final_scaler"].transform(X_frame),
        columns=fitted["columns"],
    )


def fit_pca_reducer(X_train, n_components):
    max_centered_rank = min(X_train.shape[1], X_train.shape[0] - 1)
    if n_components > max_centered_rank:
        raise ValueError(
            f"PCA-{n_components} requer posto centralizado >= {n_components}, "
            f"mas este treino permite no máximo {max_centered_rank}."
        )
    reducer = PCA(n_components=n_components, svd_solver="full")
    values = reducer.fit_transform(X_train)
    columns = [f"PC{i:03d}" for i in range(1, n_components + 1)]
    return {"pca": reducer, "columns": columns}, pd.DataFrame(values, columns=columns)


def transform_pca(fitted, X):
    return pd.DataFrame(
        fitted["pca"].transform(X),
        columns=fitted["columns"],
    ).reset_index(drop=True)


def _positive_class_shap_matrix(shap_output, n_samples, n_features):
    values = getattr(shap_output, "values", shap_output)
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        if values.shape[:2] == (n_samples, n_features):
            values = values[:, :, -1]
        elif values.shape[1:] == (n_samples, n_features):
            values = values[-1]
        else:
            raise ValueError(f"Formato SHAP 3D não reconhecido: {values.shape}")
    if values.shape != (n_samples, n_features):
        raise ValueError(
            f"Formato SHAP inesperado: {values.shape}; "
            f"esperado {(n_samples, n_features)}"
        )
    return values


def fit_shap_ranker(X_train, y_train, max_features=36):
    if max_features > X_train.shape[1]:
        raise ValueError(
            f"SHAP-{max_features} solicitado, mas o preprocessing gerou "
            f"apenas {X_train.shape[1]} colunas."
        )
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    shap_output = shap.TreeExplainer(model)(X_train, check_additivity=False)
    shap_matrix = _positive_class_shap_matrix(
        shap_output,
        len(X_train),
        X_train.shape[1],
    )
    importance = pd.Series(
        np.abs(shap_matrix).mean(axis=0),
        index=X_train.columns,
    ).sort_values(ascending=False, kind="mergesort")
    return {
        "model": model,
        "ranking": importance.index.tolist(),
        "importance": importance,
    }


def transform_shap_top(fitted, X, n_features):
    selected = fitted["ranking"][:n_features]
    return X.loc[:, selected].copy().reset_index(drop=True)


def fit_representations(X_train_raw, y_train, include_pca36):
    preprocessing, X_all = fit_preprocessing(X_train_raw)

    max_pca_components = 36 if include_pca36 else 12
    pca, X_pca_max = fit_pca_reducer(X_all, max_pca_components)
    shap_ranker = fit_shap_ranker(
        X_all,
        y_train.reset_index(drop=True),
        max_features=36,
    )

    representations = {
        "all_features": X_all.reset_index(drop=True),
        "pca12": X_pca_max.iloc[:, :12].copy().reset_index(drop=True),
        "shap12": transform_shap_top(shap_ranker, X_all, 12),
        "shap36": transform_shap_top(shap_ranker, X_all, 36),
    }
    if include_pca36:
        representations["pca36"] = X_pca_max.copy().reset_index(drop=True)

    fitted = {
        "preprocessing": preprocessing,
        "pca": pca,
        "shap": shap_ranker,
        "include_pca36": include_pca36,
    }
    return fitted, representations


def transform_representations(fitted, X_raw):
    X_all = transform_preprocessing(fitted["preprocessing"], X_raw)
    X_pca_max = transform_pca(fitted["pca"], X_all)
    representations = {
        "all_features": X_all.reset_index(drop=True),
        "pca12": X_pca_max.iloc[:, :12].copy().reset_index(drop=True),
        "shap12": transform_shap_top(fitted["shap"], X_all, 12),
        "shap36": transform_shap_top(fitted["shap"], X_all, 36),
    }
    if fitted["include_pca36"]:
        representations["pca36"] = X_pca_max.copy().reset_index(drop=True)
    return representations


## 4. Funções comuns: tuning, métricas, caches e estatística


In [4]:
CLASSIFIER_GRID = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}


def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba)
    roc_auc = (
        roc_auc_score(y_true, y_proba)
        if np.unique(y_true).size == 2 else np.nan
    )
    pr_auc = (
        average_precision_score(y_true, y_proba)
        if np.any(y_true == 1) else np.nan
    )
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
    }


def candidate_key(params):
    return json.dumps(params, sort_keys=True)


def fit_and_score_classifier(
    X_train,
    y_train,
    X_validation,
    y_validation,
    params,
):
    classifier = GradientBoostingClassifier(
        random_state=RANDOM_STATE,
        **params,
    )
    classifier.fit(X_train, y_train)
    probabilities = classifier.predict_proba(X_validation)[:, 1]
    return average_precision_score(y_validation, probabilities)


def prepare_classical_benchmark(
    X,
    y,
    outer_folds,
    sample_label,
    representations,
    include_pca36,
):
    candidates = list(ParameterGrid(CLASSIFIER_GRID))
    cached_folds = []
    fold_rows = []
    tuning_rows = []

    for outer_fold, (train_idx, test_idx) in enumerate(outer_folds, 1):
        print(
            f"[{sample_label}/clássico] outer fold "
            f"{outer_fold}/{len(outer_folds)}"
        )
        X_outer_train_raw = X.iloc[train_idx].copy()
        X_outer_test_raw = X.iloc[test_idx].copy()
        y_outer_train = y.iloc[train_idx].reset_index(drop=True)
        y_outer_test = y.iloc[test_idx].reset_index(drop=True)

        inner_folds = []
        scores = {
            representation: {
                candidate_key(params): [] for params in candidates
            }
            for representation in representations
        }

        inner_cv = TimeSeriesSplit(n_splits=INNER_SPLITS)
        for inner_fold, (inner_train_idx, inner_validation_idx) in enumerate(
            inner_cv.split(X_outer_train_raw), 1
        ):
            X_inner_train_raw = X_outer_train_raw.iloc[inner_train_idx].copy()
            X_inner_validation_raw = X_outer_train_raw.iloc[inner_validation_idx].copy()
            y_inner_train = y_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            y_inner_validation = y_outer_train.iloc[inner_validation_idx].reset_index(drop=True)

            fitted_stack, train_representations = fit_representations(
                X_inner_train_raw,
                y_inner_train,
                include_pca36=include_pca36,
            )
            validation_representations = transform_representations(
                fitted_stack,
                X_inner_validation_raw,
            )

            for representation in representations:
                for params in candidates:
                    key = candidate_key(params)
                    scores[representation][key].append(
                        fit_and_score_classifier(
                            train_representations[representation],
                            y_inner_train,
                            validation_representations[representation],
                            y_inner_validation,
                            params,
                        )
                    )

            inner_folds.append({
                "inner_fold": inner_fold,
                "train_representations": train_representations,
                "validation_representations": validation_representations,
                "y_train": y_inner_train,
                "y_validation": y_inner_validation,
            })

        best_params = {}
        for representation in representations:
            mean_scores = {
                key: float(np.mean(values))
                for key, values in scores[representation].items()
            }
            best_key = max(mean_scores, key=mean_scores.get)
            best_params[representation] = json.loads(best_key)
            for key, values in scores[representation].items():
                tuning_rows.append({
                    "sample": sample_label,
                    "outer_fold": outer_fold,
                    "representation": representation,
                    "params": key,
                    "mean_inner_average_precision": np.mean(values),
                    "std_inner_average_precision": np.std(values, ddof=1),
                })

        fitted_outer, outer_train_representations = fit_representations(
            X_outer_train_raw,
            y_outer_train,
            include_pca36=include_pca36,
        )
        outer_test_representations = transform_representations(
            fitted_outer,
            X_outer_test_raw,
        )

        classic_metrics = {}
        for representation in representations:
            params = best_params[representation]
            classifier = GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                **params,
            )
            classifier.fit(
                outer_train_representations[representation],
                y_outer_train,
            )
            prediction = classifier.predict(
                outer_test_representations[representation]
            )
            probability = classifier.predict_proba(
                outer_test_representations[representation]
            )[:, 1]
            metrics = compute_metrics(
                y_outer_test,
                prediction,
                probability,
            )
            classic_metrics[representation] = metrics
            fold_rows.append({
                "sample": sample_label,
                "representation": representation,
                "outer_fold": outer_fold,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "test_start_position": int(test_idx[0]),
                "test_end_position": int(test_idx[-1]),
                "best_params": json.dumps(params, sort_keys=True),
                **metrics,
            })

        cached_folds.append({
            "outer_fold": outer_fold,
            "train_idx": train_idx.copy(),
            "test_idx": test_idx.copy(),
            "inner_folds": inner_folds,
            "outer_train_representations": outer_train_representations,
            "outer_test_representations": outer_test_representations,
            "y_outer_train": y_outer_train,
            "y_outer_test": y_outer_test,
            "best_params": best_params,
            "classic_metrics": classic_metrics,
        })

        checkpoint = {
            "sample": sample_label,
            "fold_results": pd.DataFrame(fold_rows),
            "tuning_results": pd.DataFrame(tuning_rows),
        }
        with open(
            CHECKPOINT_DIR / f"classical_benchmark_{sample_label}_fold_{outer_fold:02d}.pkl",
            "wb",
        ) as handle:
            pickle.dump(checkpoint, handle)

    return {
        "sample": sample_label,
        "representations": tuple(representations),
        "folds": cached_folds,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }


In [5]:
def _mean_std_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    mean = float(np.mean(values)) if n else np.nan
    std = float(np.std(values, ddof=1)) if n > 1 else np.nan
    if n > 1:
        critical = student_t.ppf((1 + confidence) / 2, df=n - 1)
        margin = critical * std / np.sqrt(n)
        return n, mean, std, mean - margin, mean + margin
    return n, mean, std, np.nan, np.nan


def summarize_classical_results(fold_results):
    rows = []
    for (sample, representation), group in fold_results.groupby(
        ["sample", "representation"], sort=False
    ):
        for metric in METRICS:
            n, mean, std, ci_low, ci_high = _mean_std_ci(group[metric])
            rows.append({
                "sample": sample,
                "representation": representation,
                "metric": metric,
                "n_folds": n,
                "mean": mean,
                "std": std,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
            })
    return pd.DataFrame(rows)


def _comparison_status(delta_mean, p_value):
    significant = bool(p_value < 0.05) if np.isfinite(p_value) else False
    if np.isclose(delta_mean, 0):
        return "Sem alteração"
    if significant and delta_mean > 0:
        return "✅ MELHORA SIGNIFICATIVA"
    if significant and delta_mean < 0:
        return "⚠️ PIORA SIGNIFICATIVA"
    if delta_mean > 0:
        return "↑ Melhora (não sig.)"
    return "↓ Piora (não sig.)"


def classical_statistical_analysis(
    fold_results,
    title,
    baseline="all_features",
    n_bootstrap=2_000,
):
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    sample = fold_results["sample"].iloc[0]
    challengers = [
        value for value in fold_results["representation"].drop_duplicates()
        if value != baseline
    ]

    for challenger in challengers:
        for metric in METRICS:
            paired = fold_results.pivot(
                index="outer_fold",
                columns="representation",
                values=metric,
            )[[baseline, challenger]].dropna()
            base = paired[baseline].to_numpy(dtype=float)
            case = paired[challenger].to_numpy(dtype=float)
            delta = case - base
            if len(delta) < 2:
                statistic, p_value = np.nan, np.nan
            elif np.allclose(delta, 0):
                statistic, p_value = 0.0, 1.0
            else:
                statistic, p_value = wilcoxon(case, base, alternative="two-sided")
            indices = rng.integers(0, len(delta), size=(n_bootstrap, len(delta)))
            bootstrap_means = delta[indices].mean(axis=1)
            ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
            delta_mean = float(np.mean(delta))
            rows.append({
                "Amostra": sample,
                "Comparação": f"{baseline} vs {challenger}",
                "Métrica": metric,
                "Folds": len(delta),
                "Média Base": float(np.mean(base)),
                "Média Caso": float(np.mean(case)),
                "Δ Médio": delta_mean,
                "IC95 low": float(ci_low),
                "IC95 high": float(ci_high),
                "Estatística W": float(statistic) if np.isfinite(statistic) else np.nan,
                "p-valor": float(p_value) if np.isfinite(p_value) else np.nan,
                "Status": _comparison_status(delta_mean, p_value),
            })

    result = pd.DataFrame(rows)
    print(f"\n{'=' * 96}\n  {title}\n{'=' * 96}")
    if result.empty:
        print("Nenhuma comparação pareada disponível.")
        return result
    table = result.copy()
    table["IC95% Δ"] = table.apply(
        lambda row: f"[{row['IC95 low']:.4f}, {row['IC95 high']:.4f}]",
        axis=1,
    )
    for column in ["Média Base", "Média Caso", "Δ Médio", "p-valor"]:
        table[column] = table[column].round(4)
    print(table[[
        "Comparação", "Métrica", "Média Base", "Média Caso",
        "Δ Médio", "IC95% Δ", "p-valor", "Status",
    ]].to_string(index=False))
    return result


def summarize_quantum_results(fold_results):
    rows = []
    for (config, scenario), group in fold_results.groupby(
        ["config", "scenario"], sort=False
    ):
        for metric in METRICS:
            n, mean, std, ci_low, ci_high = _mean_std_ci(group[metric])
            rows.append({
                "config": config,
                "scenario": scenario,
                "metric": metric,
                "n_folds": n,
                "mean": mean,
                "std": std,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
            })
    return pd.DataFrame(rows)


def quantum_statistical_analysis(
    fold_results,
    title,
    n_bootstrap=2_000,
):
    rng = np.random.default_rng(RANDOM_STATE)
    comparisons = [
        ("classica", "quantum_only", "Clássico vs Quantum-Only"),
        ("classica", "hibrida", "Clássico vs Híbrido"),
    ]
    rows = []
    for metric in METRICS:
        paired = fold_results.pivot(
            index="outer_fold",
            columns="scenario",
            values=metric,
        )
        for baseline, challenger, label in comparisons:
            pair = paired[[baseline, challenger]].dropna()
            base = pair[baseline].to_numpy(dtype=float)
            case = pair[challenger].to_numpy(dtype=float)
            delta = case - base
            if len(delta) < 2:
                statistic, p_value = np.nan, np.nan
            elif np.allclose(delta, 0):
                statistic, p_value = 0.0, 1.0
            else:
                statistic, p_value = wilcoxon(case, base, alternative="two-sided")
            indices = rng.integers(0, len(delta), size=(n_bootstrap, len(delta)))
            bootstrap_means = delta[indices].mean(axis=1)
            ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
            delta_mean = float(np.mean(delta))
            rows.append({
                "Comparação": label,
                "Métrica": metric,
                "Folds": len(delta),
                "Média Base": float(np.mean(base)),
                "Média Caso": float(np.mean(case)),
                "Δ Médio": delta_mean,
                "IC95 low": float(ci_low),
                "IC95 high": float(ci_high),
                "Estatística W": float(statistic) if np.isfinite(statistic) else np.nan,
                "p-valor": float(p_value) if np.isfinite(p_value) else np.nan,
                "Status": _comparison_status(delta_mean, p_value),
            })

    result = pd.DataFrame(rows)
    print(f"\n{'=' * 96}\n  {title}\n{'=' * 96}")
    table = result.copy()
    table["IC95% Δ"] = table.apply(
        lambda row: f"[{row['IC95 low']:.4f}, {row['IC95 high']:.4f}]",
        axis=1,
    )
    for column in ["Média Base", "Média Caso", "Δ Médio", "p-valor"]:
        table[column] = table[column].round(4)
    print(table[[
        "Comparação", "Métrica", "Média Base", "Média Caso",
        "Δ Médio", "IC95% Δ", "p-valor", "Status",
    ]].to_string(index=False))
    return result


# PARTE I — 1.000 amostras

Esta parte usa uma amostra estratificada de 1.000 registros. O benchmark clássico contém `all_features`, `pca12`, `shap12` e `shap36`. A camada quântica usa exclusivamente PCA-12.


In [6]:
sampled_1000 = stratified_temporal_sample(df, n_samples=1_000)
X_1000 = sampled_1000[FEATURE_COLS].copy()
y_1000 = sampled_1000[TARGET_COL].astype(int).copy()
OUTER_FOLDS_1000 = make_outer_folds(X_1000)

print(f"Amostra 1.000: {X_1000.shape}")
print(f"Distribuição: {y_1000.value_counts(normalize=True).sort_index().round(4).to_dict()}")
print(f"Período: {sampled_1000['period_start'].min()} → {sampled_1000['period_start'].max()}")
display(describe_nested_sizes(OUTER_FOLDS_1000))

assert min(
    min(sizes)
    for sizes in describe_nested_sizes(OUTER_FOLDS_1000)["inner_train_sizes"]
) >= 13, "A menor janela deve permitir 12 componentes após centralização."


Amostra 1.000: (1000, 36)
Distribuição: {0: 0.32, 1: 0.68}
Período: 1952-01-18 02:54:36+00:00 → 2023-03-17 02:54:36+00:00


,outer_fold,outer_train,outer_test,inner_train_sizes
0,1,100,90,"[25, 50, 75]"
1,2,190,90,"[49, 96, 143]"
2,3,280,90,"[70, 140, 210]"
3,4,370,90,"[94, 186, 278]"
4,5,460,90,"[115, 230, 345]"
5,6,550,90,"[139, 276, 413]"
6,7,640,90,"[160, 320, 480]"
7,8,730,90,"[184, 366, 548]"
8,9,820,90,"[205, 410, 615]"
9,10,910,90,"[229, 456, 683]"


## Parte I.A — Benchmark clássico, 1.000 amostras

Os quatro pipelines usam os mesmos outer/inner folds e fazem tuning independente pelo mesmo grid.


In [7]:
REPRESENTATIONS_1000 = (
    "all_features",
    "pca12",
    "shap12",
    "shap36",
)
BENCHMARK_KEY_1000 = (
    id(X_1000),
    RANDOM_STATE,
    REPRESENTATIONS_1000,
    tuple((name, tuple(values)) for name, values in sorted(CLASSIFIER_GRID.items())),
)

benchmark_is_current = (
    "CLASSICAL_BENCHMARK_N1000" in globals()
    and CLASSICAL_BENCHMARK_N1000.get("cache_key") == BENCHMARK_KEY_1000
)

if benchmark_is_current:
    print("Benchmark clássico n1000 já existe; reutilizando cache.")
else:
    CLASSICAL_BENCHMARK_N1000 = prepare_classical_benchmark(
        X=X_1000,
        y=y_1000,
        outer_folds=OUTER_FOLDS_1000,
        sample_label="n1000",
        representations=REPRESENTATIONS_1000,
        include_pca36=False,
    )
    CLASSICAL_BENCHMARK_N1000["cache_key"] = BENCHMARK_KEY_1000


[n1000/clássico] outer fold 1/10
[n1000/clássico] outer fold 2/10
[n1000/clássico] outer fold 3/10
[n1000/clássico] outer fold 4/10
[n1000/clássico] outer fold 5/10
[n1000/clássico] outer fold 6/10
[n1000/clássico] outer fold 7/10
[n1000/clássico] outer fold 8/10
[n1000/clássico] outer fold 9/10
[n1000/clássico] outer fold 10/10


In [8]:
CLASSICAL_SUMMARY_N1000 = summarize_classical_results(
    CLASSICAL_BENCHMARK_N1000["fold_results"]
)
CLASSICAL_STATS_N1000 = classical_statistical_analysis(
    CLASSICAL_BENCHMARK_N1000["fold_results"],
    title="BENCHMARK CLÁSSICO — 1.000 AMOSTRAS — 10 OUTER FOLDS",
)

display(CLASSICAL_BENCHMARK_N1000["fold_results"])
display(CLASSICAL_SUMMARY_N1000)



  BENCHMARK CLÁSSICO — 1.000 AMOSTRAS — 10 OUTER FOLDS
            Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
 all_features vs pca12     ROC-AUC      0.8107      0.7573  -0.0534  [-0.0991, 0.0024]   0.0840     ↓ Piora (não sig.)
 all_features vs pca12      PR-AUC      0.8788      0.8660  -0.0128  [-0.0477, 0.0423]   0.0840     ↓ Piora (não sig.)
 all_features vs pca12    Accuracy      0.7567      0.6811  -0.0756 [-0.1178, -0.0400]   0.0039 ⚠️ PIORA SIGNIFICATIVA
 all_features vs pca12   Precision      0.8170      0.7870  -0.0300 [-0.0560, -0.0074]   0.0488 ⚠️ PIORA SIGNIFICATIVA
 all_features vs pca12      Recall      0.8030      0.7129  -0.0901 [-0.1730, -0.0259]   0.0195 ⚠️ PIORA SIGNIFICATIVA
 all_features vs pca12          F1      0.8081      0.7364  -0.0716 [-0.1294, -0.0307]   0.0039 ⚠️ PIORA SIGNIFICATIVA
 all_features vs pca12 Specificity      0.6689      0.6175  -0.0514  [-0.1269, 0.0239]   0.2031     ↓ Piora (nã

,sample,representation,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,n1000,all_features,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997531,0.944444,0.987342,0.951220,0.968944,0.875000
1,n1000,pca12,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.850610,0.971805,0.900000,0.974026,0.914634,0.943396,0.750000
2,n1000,shap12,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997575,0.944444,0.975309,0.963415,0.969325,0.750000
3,n1000,shap36,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.972561,0.997396,0.944444,0.987342,0.951220,0.968944,0.875000
4,n1000,all_features,2,190,90,190,279,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.870694,0.933492,0.766667,0.839286,0.796610,0.817391,0.709677
5,n1000,pca12,2,190,90,190,279,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.845271,0.899316,0.555556,0.880000,0.372881,0.523810,0.903226
6,n1000,shap12,2,190,90,190,279,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.863587,0.911120,0.788889,0.884615,0.779661,0.828829,0.806452
7,n1000,shap36,2,190,90,190,279,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.864680,0.913887,0.766667,0.851852,0.779661,0.814159,0.741935
8,n1000,all_features,3,280,90,280,369,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.939001,0.980024,0.866667,0.910448,0.910448,0.910448,0.739130
9,n1000,pca12,3,280,90,280,369,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.924724,0.976402,0.877778,0.911765,0.925373,0.918519,0.739130


,sample,representation,metric,n_folds,mean,std,ci95_low,ci95_high
0,n1000,all_features,ROC-AUC,10,0.810732,0.136739,0.712914,0.908549
1,n1000,all_features,PR-AUC,10,0.878824,0.132750,0.783860,0.973788
2,n1000,all_features,Accuracy,10,0.756667,0.103631,0.682534,0.830800
3,n1000,all_features,Precision,10,0.816983,0.114598,0.735004,0.898962
4,n1000,all_features,Recall,10,0.802960,0.096739,0.733757,0.872163
5,n1000,all_features,F1,10,0.808072,0.098562,0.737565,0.878579
6,n1000,all_features,Specificity,10,0.668859,0.111946,0.588778,0.748940
7,n1000,pca12,ROC-AUC,10,0.757311,0.101738,0.684532,0.830090
8,n1000,pca12,PR-AUC,10,0.865978,0.098874,0.795248,0.936708
9,n1000,pca12,Accuracy,10,0.681111,0.121947,0.593876,0.768347


## Parte I.B — Heisenberg 13q × PCA-12, GPU

Treze qubits formam uma cadeia com 12 arestas. As 12 componentes PCA ocupam exatamente um bloco. O circuito é ajustado uma única vez; as features quânticas são pré-calculadas para as representações PCA-12 específicas de cada fold.


In [10]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "heisenberg_13q_pca12_n1000_gpu"
N_QUBITS = 13
N_PCA_COMPONENTS = 12
HEISENBERG_R = 2
HEISENBERG_ALPHA = 0.1
HEISENBERG_SHOTS = SHOTS
USE_TANH_SCALING = True
MEASURE_2LOCAL_DIAGONAL = False
CLASSICAL_BENCHMARK = CLASSICAL_BENCHMARK_N1000
PCA_REPRESENTATION = "pca12"
# ────────────────────────────────────────────────────────────────────

HEISENBERG_CACHE_KEY_13Q = (
    RUN_LABEL,
    id(CLASSICAL_BENCHMARK),
    RANDOM_STATE,
    N_QUBITS,
    N_PCA_COMPONENTS,
    HEISENBERG_R,
    HEISENBERG_ALPHA,
    HEISENBERG_SHOTS,
    USE_TANH_SCALING,
    MEASURE_2LOCAL_DIAGONAL,
)
cache_is_current = (
    "HEISENBERG_PCA_CACHE_13Q_N1000" in globals()
    and HEISENBERG_PCA_CACHE_13Q_N1000.get("cache_key")
    == HEISENBERG_CACHE_KEY_13Q
)

if cache_is_current:
    print("Cache Heisenberg 13q/PCA-12/n1000 já existe; reutilizando.")
else:
    HEISENBERG_PQFM_13Q = HeisenbergProjectiveQFM(
        name_file=RUN_LABEL,
        seed=RANDOM_STATE,
        ideal=True,
        simulation=True,
        fakebackend=False,
        shots=HEISENBERG_SHOTS,
        q_enc=N_QUBITS,
        R=HEISENBERG_R,
        alpha=HEISENBERG_ALPHA,
        use_tanh_scaling=USE_TANH_SCALING,
        measure_2local_diagonal=MEASURE_2LOCAL_DIAGONAL,
        use_gpu_statevector=True,
        statevector_device="GPU",
        output_root=f"outputs/pqfm/{RUN_LABEL}",
    )
    print("Preparando uma única vez o circuito Heisenberg 13q na GPU...")
    HEISENBERG_PQFM_13Q.fit(
        np.zeros((1, N_PCA_COMPONENTS), dtype=float)
    )
    assert HEISENBERG_PQFM_13Q.theta_info["features_per_block"] == 12
    assert HEISENBERG_PQFM_13Q.theta_info["num_blocks"] == 1
    assert HEISENBERG_PQFM_13Q.theta_info["total_slots"] == 12

    quantum_folds = []
    transform_counter = 0
    transform_total = len(CLASSICAL_BENCHMARK["folds"]) * (INNER_SPLITS + 1)

    for classic_fold in CLASSICAL_BENCHMARK["folds"]:
        outer_fold = classic_fold["outer_fold"]
        inner_quantum = []
        for inner_data in classic_fold["inner_folds"]:
            X_train_pca = inner_data["train_representations"][PCA_REPRESENTATION]
            X_validation_pca = inner_data["validation_representations"][PCA_REPRESENTATION]
            n_train = len(X_train_pca)
            X_all_pca = pd.concat(
                [X_train_pca, X_validation_pca],
                ignore_index=True,
            )
            transform_counter += 1
            print(
                f"[Heisenberg-13q {transform_counter:02d}/{transform_total:02d}] "
                f"outer={outer_fold}/10 | inner={inner_data['inner_fold']}/3 | "
                f"amostras={len(X_all_pca)} | GPU"
            )
            started_at = time.perf_counter()
            Xq_values = HEISENBERG_PQFM_13Q.transform(X_all_pca)
            q_columns = [f"heisenberg_q_{i}" for i in range(Xq_values.shape[1])]
            Xq_all = pd.DataFrame(Xq_values, columns=q_columns)
            inner_quantum.append({
                "inner_fold": inner_data["inner_fold"],
                "X_train_quantum": Xq_all.iloc[:n_train].reset_index(drop=True),
                "X_validation_quantum": Xq_all.iloc[n_train:].reset_index(drop=True),
            })
            print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        X_outer_train_pca = classic_fold["outer_train_representations"][PCA_REPRESENTATION]
        X_outer_test_pca = classic_fold["outer_test_representations"][PCA_REPRESENTATION]
        n_outer_train = len(X_outer_train_pca)
        X_outer_all_pca = pd.concat(
            [X_outer_train_pca, X_outer_test_pca],
            ignore_index=True,
        )
        transform_counter += 1
        print(
            f"[Heisenberg-13q {transform_counter:02d}/{transform_total:02d}] "
            f"outer={outer_fold}/10 | OUTER | "
            f"amostras={len(X_outer_all_pca)} | GPU"
        )
        started_at = time.perf_counter()
        Xq_outer_values = HEISENBERG_PQFM_13Q.transform(X_outer_all_pca)
        q_columns = [
            f"heisenberg_q_{i}" for i in range(Xq_outer_values.shape[1])
        ]
        Xq_outer_all = pd.DataFrame(Xq_outer_values, columns=q_columns)
        print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        quantum_folds.append({
            "classic_fold": classic_fold,
            "inner_quantum": inner_quantum,
            "X_outer_train_quantum": Xq_outer_all.iloc[:n_outer_train].reset_index(drop=True),
            "X_outer_test_quantum": Xq_outer_all.iloc[n_outer_train:].reset_index(drop=True),
        })

    HEISENBERG_PCA_CACHE_13Q_N1000 = {
        "cache_key": HEISENBERG_CACHE_KEY_13Q,
        "config": RUN_LABEL,
        "pca_representation": PCA_REPRESENTATION,
        "n_qubits": N_QUBITS,
        "n_input_features": N_PCA_COMPONENTS,
        "n_blocks": HEISENBERG_PQFM_13Q.theta_info["num_blocks"],
        "n_quantum_features": len(q_columns),
        "folds": quantum_folds,
    }
    print(
        f"Cache concluído: {len(quantum_folds)} outer folds | "
        f"{len(q_columns)} features quânticas por amostra."
    )


Preparando uma única vez o circuito Heisenberg 13q na GPU...
[Heisenberg-13q 01/40] outer=1/10 | inner=1/3 | amostras=50 | GPU
  concluído em 6.6s
[Heisenberg-13q 02/40] outer=1/10 | inner=2/3 | amostras=75 | GPU
  concluído em 10.6s
[Heisenberg-13q 03/40] outer=1/10 | inner=3/3 | amostras=100 | GPU
  concluído em 14.1s
[Heisenberg-13q 04/40] outer=1/10 | OUTER | amostras=190 | GPU
  concluído em 28.7s
[Heisenberg-13q 05/40] outer=2/10 | inner=1/3 | amostras=96 | GPU
  concluído em 15.2s
[Heisenberg-13q 06/40] outer=2/10 | inner=2/3 | amostras=143 | GPU
  concluído em 23.2s
[Heisenberg-13q 07/40] outer=2/10 | inner=3/3 | amostras=190 | GPU
  concluído em 30.6s
[Heisenberg-13q 08/40] outer=2/10 | OUTER | amostras=280 | GPU
  concluído em 46.3s
[Heisenberg-13q 09/40] outer=3/10 | inner=1/3 | amostras=140 | GPU
  concluído em 24.4s
[Heisenberg-13q 10/40] outer=3/10 | inner=2/3 | amostras=210 | GPU
  concluído em 37.4s
[Heisenberg-13q 11/40] outer=3/10 | inner=3/3 | amostras=280 | GPU
  co

In [11]:
# Nested tuning e avaliação. Esta célula não chama a PQFM.
RUN_LABEL = HEISENBERG_PCA_CACHE_13Q_N1000["config"]
PCA_REPRESENTATION = HEISENBERG_PCA_CACHE_13Q_N1000["pca_representation"]
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []

for quantum_fold in HEISENBERG_PCA_CACHE_13Q_N1000["folds"]:
    classic_fold = quantum_fold["classic_fold"]
    outer_fold = classic_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] tuning/classificação outer fold {outer_fold}/10")
    scores = {
        "quantum_only": {candidate_key(params): [] for params in candidates},
        "hibrida": {candidate_key(params): [] for params in candidates},
    }

    for inner_data, inner_quantum in zip(
        classic_fold["inner_folds"],
        quantum_fold["inner_quantum"],
        strict=True,
    ):
        X_train_classical = inner_data["train_representations"]["all_features"]
        X_validation_classical = inner_data["validation_representations"]["all_features"]
        X_train_quantum = inner_quantum["X_train_quantum"]
        X_validation_quantum = inner_quantum["X_validation_quantum"]
        X_train_hybrid = pd.concat(
            [X_train_classical.reset_index(drop=True), X_train_quantum], axis=1
        )
        X_validation_hybrid = pd.concat(
            [X_validation_classical.reset_index(drop=True), X_validation_quantum], axis=1
        )

        for params in candidates:
            key = candidate_key(params)
            scores["quantum_only"][key].append(
                fit_and_score_classifier(
                    X_train_quantum,
                    inner_data["y_train"],
                    X_validation_quantum,
                    inner_data["y_validation"],
                    params,
                )
            )
            scores["hibrida"][key].append(
                fit_and_score_classifier(
                    X_train_hybrid,
                    inner_data["y_train"],
                    X_validation_hybrid,
                    inner_data["y_validation"],
                    params,
                )
            )

    best_params = {}
    for scenario in ("quantum_only", "hibrida"):
        mean_scores = {
            key: float(np.mean(values))
            for key, values in scores[scenario].items()
        }
        best_key = max(mean_scores, key=mean_scores.get)
        best_params[scenario] = json.loads(best_key)
        for key, values in scores[scenario].items():
            tuning_rows.append({
                "config": RUN_LABEL,
                "outer_fold": outer_fold,
                "scenario": scenario,
                "params": key,
                "mean_inner_average_precision": np.mean(values),
                "std_inner_average_precision": np.std(values, ddof=1),
            })

    fold_rows.append({
        "config": RUN_LABEL,
        "scenario": "classica",
        "outer_fold": outer_fold,
        "n_train": len(classic_fold["train_idx"]),
        "n_test": len(classic_fold["test_idx"]),
        "test_start_position": int(classic_fold["test_idx"][0]),
        "test_end_position": int(classic_fold["test_idx"][-1]),
        "best_params": json.dumps(
            classic_fold["best_params"]["all_features"], sort_keys=True
        ),
        **classic_fold["classic_metrics"]["all_features"],
    })

    X_outer_train_classical = classic_fold["outer_train_representations"]["all_features"]
    X_outer_test_classical = classic_fold["outer_test_representations"]["all_features"]
    X_outer_train_quantum = quantum_fold["X_outer_train_quantum"]
    X_outer_test_quantum = quantum_fold["X_outer_test_quantum"]
    X_outer_train_hybrid = pd.concat(
        [X_outer_train_classical.reset_index(drop=True), X_outer_train_quantum], axis=1
    )
    X_outer_test_hybrid = pd.concat(
        [X_outer_test_classical.reset_index(drop=True), X_outer_test_quantum], axis=1
    )

    for scenario, X_train_rep, X_test_rep in (
        ("quantum_only", X_outer_train_quantum, X_outer_test_quantum),
        ("hibrida", X_outer_train_hybrid, X_outer_test_hybrid),
    ):
        params = best_params[scenario]
        classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params,
        )
        classifier.fit(X_train_rep, classic_fold["y_outer_train"])
        prediction = classifier.predict(X_test_rep)
        probability = classifier.predict_proba(X_test_rep)[:, 1]
        fold_rows.append({
            "config": RUN_LABEL,
            "scenario": scenario,
            "outer_fold": outer_fold,
            "n_train": len(classic_fold["train_idx"]),
            "n_test": len(classic_fold["test_idx"]),
            "test_start_position": int(classic_fold["test_idx"][0]),
            "test_end_position": int(classic_fold["test_idx"][-1]),
            "best_params": json.dumps(params, sort_keys=True),
            **compute_metrics(
                classic_fold["y_outer_test"], prediction, probability
            ),
        })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        CHECKPOINT_DIR / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl",
        "wb",
    ) as handle:
        pickle.dump(checkpoint, handle)

HEISENBERG_13Q_PCA12_N1000_RESULT = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
print(f"Experimento {RUN_LABEL} concluído.")



[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 1/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 2/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 3/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 4/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 5/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 6/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 7/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 8/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 9/10

[heisenberg_13q_pca12_n1000_gpu] tuning/classificação outer fold 10/10
Experimento heisenberg_13q_pca12_n1000_gpu concluído.


In [12]:
HEISENBERG_13Q_SUMMARY = summarize_quantum_results(
    HEISENBERG_13Q_PCA12_N1000_RESULT["fold_results"]
)
HEISENBERG_13Q_STATS = quantum_statistical_analysis(
    HEISENBERG_13Q_PCA12_N1000_RESULT["fold_results"],
    title="HEISENBERG 13Q × PCA-12 × N1000 — 10 OUTER FOLDS",
)
display(HEISENBERG_13Q_PCA12_N1000_RESULT["fold_results"])
display(HEISENBERG_13Q_SUMMARY)



  HEISENBERG 13Q × PCA-12 × N1000 — 10 OUTER FOLDS
              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
Clássico vs Quantum-Only     ROC-AUC      0.8107      0.7659  -0.0448  [-0.0880, 0.0134]   0.1055     ↓ Piora (não sig.)
     Clássico vs Híbrido     ROC-AUC      0.8107      0.8150   0.0043  [-0.0351, 0.0585]   0.3750   ↑ Melhora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.8788      0.8714  -0.0074  [-0.0409, 0.0442]   0.1055     ↓ Piora (não sig.)
     Clássico vs Híbrido      PR-AUC      0.8788      0.8901   0.0112  [-0.0154, 0.0528]   0.3750   ↑ Melhora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7567      0.7056  -0.0511 [-0.0744, -0.0278]   0.0078 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbrido    Accuracy      0.7567      0.7478  -0.0089  [-0.0244, 0.0078]   0.3438     ↓ Piora (não sig.)
Clássico vs Quantum-Only   Precision      0.8170      0.8020  -0.0150  [-0.0347, 0.0067]   0.2754    

,config,scenario,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,heisenberg_13q_pca12_n1000_gpu,classica,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.974085,0.997531,0.944444,0.987342,0.951220,0.968944,0.875000
1,heisenberg_13q_pca12_n1000_gpu,quantum_only,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.913110,0.991237,0.877778,0.973333,0.890244,0.929936,0.750000
2,heisenberg_13q_pca12_n1000_gpu,hibrida,1,100,90,100,189,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.949695,0.994923,0.911111,0.974359,0.926829,0.950000,0.750000
3,heisenberg_13q_pca12_n1000_gpu,classica,2,190,90,190,279,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.870694,0.933492,0.766667,0.839286,0.796610,0.817391,0.709677
4,heisenberg_13q_pca12_n1000_gpu,quantum_only,2,190,90,190,279,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.752324,0.867756,0.666667,0.853659,0.593220,0.700000,0.806452
5,heisenberg_13q_pca12_n1000_gpu,hibrida,2,190,90,190,279,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.863587,0.922880,0.744444,0.875000,0.711864,0.785047,0.806452
6,heisenberg_13q_pca12_n1000_gpu,classica,3,280,90,280,369,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.939001,0.980024,0.866667,0.910448,0.910448,0.910448,0.739130
7,heisenberg_13q_pca12_n1000_gpu,quantum_only,3,280,90,280,369,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.914666,0.969360,0.877778,0.911765,0.925373,0.918519,0.739130
8,heisenberg_13q_pca12_n1000_gpu,hibrida,3,280,90,280,369,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.934458,0.977938,0.877778,0.924242,0.910448,0.917293,0.782609
9,heisenberg_13q_pca12_n1000_gpu,classica,4,370,90,370,459,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.796875,0.924385,0.722222,0.867925,0.718750,0.786325,0.730769


,config,scenario,metric,n_folds,mean,std,ci95_low,ci95_high
0,heisenberg_13q_pca12_n1000_gpu,classica,ROC-AUC,10,0.810732,0.136739,0.712914,0.908549
1,heisenberg_13q_pca12_n1000_gpu,classica,PR-AUC,10,0.878824,0.132750,0.783860,0.973788
2,heisenberg_13q_pca12_n1000_gpu,classica,Accuracy,10,0.756667,0.103631,0.682534,0.830800
3,heisenberg_13q_pca12_n1000_gpu,classica,Precision,10,0.816983,0.114598,0.735004,0.898962
4,heisenberg_13q_pca12_n1000_gpu,classica,Recall,10,0.802960,0.096739,0.733757,0.872163
5,heisenberg_13q_pca12_n1000_gpu,classica,F1,10,0.808072,0.098562,0.737565,0.878579
6,heisenberg_13q_pca12_n1000_gpu,classica,Specificity,10,0.668859,0.111946,0.588778,0.748940
7,heisenberg_13q_pca12_n1000_gpu,quantum_only,ROC-AUC,10,0.765914,0.100982,0.693676,0.838152
8,heisenberg_13q_pca12_n1000_gpu,quantum_only,PR-AUC,10,0.871385,0.097782,0.801436,0.941334
9,heisenberg_13q_pca12_n1000_gpu,quantum_only,Accuracy,10,0.705556,0.101329,0.633069,0.778042


# PARTE II — 2.000 amostras

Esta parte usa uma nova amostra estratificada de 2.000 registros. O benchmark clássico contém `all_features`, `pca12`, `pca36`, `shap12` e `shap36`. A camada quântica usa exclusivamente PCA-36.


In [9]:
sampled_2000 = stratified_temporal_sample(df, n_samples=2_000)
X_2000 = sampled_2000[FEATURE_COLS].copy()
y_2000 = sampled_2000[TARGET_COL].astype(int).copy()
OUTER_FOLDS_2000 = make_outer_folds(X_2000)

print(f"Amostra 2.000: {X_2000.shape}")
print(f"Distribuição: {y_2000.value_counts(normalize=True).sort_index().round(4).to_dict()}")
print(f"Período: {sampled_2000['period_start'].min()} → {sampled_2000['period_start'].max()}")
display(describe_nested_sizes(OUTER_FOLDS_2000))

assert min(
    min(sizes)
    for sizes in describe_nested_sizes(OUTER_FOLDS_2000)["inner_train_sizes"]
) >= 37, "A menor janela deve permitir 36 componentes após centralização."


Amostra 2.000: (2000, 36)
Distribuição: {0: 0.3195, 1: 0.6805}
Período: 1946-04-22 04:58:16+00:00 → 2023-03-25 20:57:28+00:00


,outer_fold,outer_train,outer_test,inner_train_sizes
0,1,190,181,"[49, 96, 143]"
1,2,371,181,"[95, 187, 279]"
2,3,552,181,"[138, 276, 414]"
3,4,733,181,"[184, 367, 550]"
4,5,914,181,"[230, 458, 686]"
5,6,1095,181,"[276, 549, 822]"
6,7,1276,181,"[319, 638, 957]"
7,8,1457,181,"[365, 729, 1093]"
8,9,1638,181,"[411, 820, 1229]"
9,10,1819,181,"[457, 911, 1365]"


## Parte II.A — Benchmark clássico, 2.000 amostras


In [10]:
REPRESENTATIONS_2000 = (
    "all_features",
    "pca12",
    "pca36",
    "shap12",
    "shap36",
)
BENCHMARK_KEY_2000 = (
    id(X_2000),
    RANDOM_STATE,
    REPRESENTATIONS_2000,
    tuple((name, tuple(values)) for name, values in sorted(CLASSIFIER_GRID.items())),
)

benchmark_is_current = (
    "CLASSICAL_BENCHMARK_N2000" in globals()
    and CLASSICAL_BENCHMARK_N2000.get("cache_key") == BENCHMARK_KEY_2000
)

if benchmark_is_current:
    print("Benchmark clássico n2000 já existe; reutilizando cache.")
else:
    CLASSICAL_BENCHMARK_N2000 = prepare_classical_benchmark(
        X=X_2000,
        y=y_2000,
        outer_folds=OUTER_FOLDS_2000,
        sample_label="n2000",
        representations=REPRESENTATIONS_2000,
        include_pca36=True,
    )
    CLASSICAL_BENCHMARK_N2000["cache_key"] = BENCHMARK_KEY_2000


[n2000/clássico] outer fold 1/10
[n2000/clássico] outer fold 2/10
[n2000/clássico] outer fold 3/10
[n2000/clássico] outer fold 4/10
[n2000/clássico] outer fold 5/10
[n2000/clássico] outer fold 6/10
[n2000/clássico] outer fold 7/10
[n2000/clássico] outer fold 8/10
[n2000/clássico] outer fold 9/10
[n2000/clássico] outer fold 10/10


In [11]:
CLASSICAL_SUMMARY_N2000 = summarize_classical_results(
    CLASSICAL_BENCHMARK_N2000["fold_results"]
)
CLASSICAL_STATS_N2000 = classical_statistical_analysis(
    CLASSICAL_BENCHMARK_N2000["fold_results"],
    title="BENCHMARK CLÁSSICO — 2.000 AMOSTRAS — 10 OUTER FOLDS",
)

display(CLASSICAL_BENCHMARK_N2000["fold_results"])
display(CLASSICAL_SUMMARY_N2000)



  BENCHMARK CLÁSSICO — 2.000 AMOSTRAS — 10 OUTER FOLDS
            Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
 all_features vs pca12     ROC-AUC      0.8437      0.7958  -0.0479 [-0.0723, -0.0223]   0.0195 ⚠️ PIORA SIGNIFICATIVA
 all_features vs pca12      PR-AUC      0.9080      0.8844  -0.0236 [-0.0398, -0.0066]   0.0645     ↓ Piora (não sig.)
 all_features vs pca12    Accuracy      0.7707      0.7298  -0.0409  [-0.0906, 0.0011]   0.1641     ↓ Piora (não sig.)
 all_features vs pca12   Precision      0.8308      0.8064  -0.0244 [-0.0491, -0.0010]   0.1602     ↓ Piora (não sig.)
 all_features vs pca12      Recall      0.8123      0.7822  -0.0301  [-0.1143, 0.0412]   0.9219     ↓ Piora (não sig.)
 all_features vs pca12          F1      0.8205      0.7867  -0.0338  [-0.0872, 0.0090]   0.5566     ↓ Piora (não sig.)
 all_features vs pca12 Specificity      0.6785      0.6272  -0.0513  [-0.1274, 0.0165]   0.3750     ↓ Piora (nã

,sample,representation,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,n2000,all_features,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.980402,0.997854,0.966851,0.981595,0.981595,0.981595,0.833333
1,n2000,pca12,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.980743,0.997903,0.955801,0.981366,0.969325,0.975309,0.833333
2,n2000,pca36,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 5, ""n_est...",0.989264,0.998466,0.972376,1.000000,0.969325,0.984424,1.000000
3,n2000,shap12,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.979721,0.997806,0.961326,0.975610,0.981595,0.978593,0.777778
4,n2000,shap36,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.980743,0.997921,0.966851,0.981595,0.981595,0.981595,0.833333
5,n2000,all_features,2,371,181,371,551,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.921175,0.945414,0.861878,0.913043,0.875000,0.893617,0.836066
6,n2000,pca12,2,371,181,371,551,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.874658,0.912420,0.651934,0.901408,0.533333,0.670157,0.885246
7,n2000,pca36,2,371,181,371,551,"{""learning_rate"": 0.05, ""max_depth"": 5, ""n_est...",0.828825,0.890933,0.662983,0.855422,0.591667,0.699507,0.803279
8,n2000,shap12,2,371,181,371,551,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.910451,0.940618,0.828729,0.880342,0.858333,0.869198,0.770492
9,n2000,shap36,2,371,181,371,551,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.920014,0.946899,0.828729,0.893805,0.841667,0.866953,0.803279


,sample,representation,metric,n_folds,mean,std,ci95_low,ci95_high
0,n2000,all_features,ROC-AUC,10,0.843733,0.081463,0.785458,0.902008
1,n2000,all_features,PR-AUC,10,0.908034,0.086030,0.846492,0.969576
2,n2000,all_features,Accuracy,10,0.770718,0.093079,0.704133,0.837303
3,n2000,all_features,Precision,10,0.830759,0.090734,0.765852,0.895666
4,n2000,all_features,Recall,10,0.812332,0.086797,0.750241,0.874423
5,n2000,all_features,F1,10,0.820493,0.085396,0.759405,0.881582
6,n2000,all_features,Specificity,10,0.678487,0.088858,0.614921,0.742052
7,n2000,pca12,ROC-AUC,10,0.795834,0.109810,0.717281,0.874388
8,n2000,pca12,PR-AUC,10,0.884422,0.084429,0.824024,0.944819
9,n2000,pca12,Accuracy,10,0.729834,0.112488,0.649365,0.810303


## Parte II.B — Heisenberg 19q × PCA-36, GPU

Dezenove qubits formam uma cadeia com 18 arestas por bloco. As 36 componentes PCA ocupam exatamente dois blocos.


In [16]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "heisenberg_19q_pca36_n2000_gpu"
N_QUBITS = 19
N_PCA_COMPONENTS = 36
HEISENBERG_R = 2
HEISENBERG_ALPHA = 0.1
HEISENBERG_SHOTS = SHOTS
USE_TANH_SCALING = True
MEASURE_2LOCAL_DIAGONAL = False
CLASSICAL_BENCHMARK = CLASSICAL_BENCHMARK_N2000
PCA_REPRESENTATION = "pca36"
# ────────────────────────────────────────────────────────────────────

HEISENBERG_CACHE_KEY_19Q = (
    RUN_LABEL,
    id(CLASSICAL_BENCHMARK),
    RANDOM_STATE,
    N_QUBITS,
    N_PCA_COMPONENTS,
    HEISENBERG_R,
    HEISENBERG_ALPHA,
    HEISENBERG_SHOTS,
    USE_TANH_SCALING,
    MEASURE_2LOCAL_DIAGONAL,
)
cache_is_current = (
    "HEISENBERG_PCA_CACHE_19Q_N2000" in globals()
    and HEISENBERG_PCA_CACHE_19Q_N2000.get("cache_key")
    == HEISENBERG_CACHE_KEY_19Q
)

if cache_is_current:
    print("Cache Heisenberg 19q/PCA-36/n2000 já existe; reutilizando.")
else:
    HEISENBERG_PQFM_19Q = HeisenbergProjectiveQFM(
        name_file=RUN_LABEL,
        seed=RANDOM_STATE,
        ideal=True,
        simulation=True,
        fakebackend=False,
        shots=HEISENBERG_SHOTS,
        q_enc=N_QUBITS,
        R=HEISENBERG_R,
        alpha=HEISENBERG_ALPHA,
        use_tanh_scaling=USE_TANH_SCALING,
        measure_2local_diagonal=MEASURE_2LOCAL_DIAGONAL,
        use_gpu_statevector=True,
        statevector_device="GPU",
        output_root=f"outputs/pqfm/{RUN_LABEL}",
    )
    print("Preparando uma única vez o circuito Heisenberg 19q na GPU...")
    HEISENBERG_PQFM_19Q.fit(
        np.zeros((1, N_PCA_COMPONENTS), dtype=float)
    )
    assert HEISENBERG_PQFM_19Q.theta_info["features_per_block"] == 18
    assert HEISENBERG_PQFM_19Q.theta_info["num_blocks"] == 2
    assert HEISENBERG_PQFM_19Q.theta_info["total_slots"] == 36

    quantum_folds = []
    transform_counter = 0
    transform_total = len(CLASSICAL_BENCHMARK["folds"]) * (INNER_SPLITS + 1)

    for classic_fold in CLASSICAL_BENCHMARK["folds"]:
        outer_fold = classic_fold["outer_fold"]
        inner_quantum = []
        for inner_data in classic_fold["inner_folds"]:
            X_train_pca = inner_data["train_representations"][PCA_REPRESENTATION]
            X_validation_pca = inner_data["validation_representations"][PCA_REPRESENTATION]
            n_train = len(X_train_pca)
            X_all_pca = pd.concat(
                [X_train_pca, X_validation_pca],
                ignore_index=True,
            )
            transform_counter += 1
            print(
                f"[Heisenberg-19q {transform_counter:02d}/{transform_total:02d}] "
                f"outer={outer_fold}/10 | inner={inner_data['inner_fold']}/3 | "
                f"amostras={len(X_all_pca)} | GPU"
            )
            started_at = time.perf_counter()
            Xq_values = HEISENBERG_PQFM_19Q.transform(X_all_pca)
            q_columns = [f"heisenberg_q_{i}" for i in range(Xq_values.shape[1])]
            Xq_all = pd.DataFrame(Xq_values, columns=q_columns)
            inner_quantum.append({
                "inner_fold": inner_data["inner_fold"],
                "X_train_quantum": Xq_all.iloc[:n_train].reset_index(drop=True),
                "X_validation_quantum": Xq_all.iloc[n_train:].reset_index(drop=True),
            })
            print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        X_outer_train_pca = classic_fold["outer_train_representations"][PCA_REPRESENTATION]
        X_outer_test_pca = classic_fold["outer_test_representations"][PCA_REPRESENTATION]
        n_outer_train = len(X_outer_train_pca)
        X_outer_all_pca = pd.concat(
            [X_outer_train_pca, X_outer_test_pca],
            ignore_index=True,
        )
        transform_counter += 1
        print(
            f"[Heisenberg-19q {transform_counter:02d}/{transform_total:02d}] "
            f"outer={outer_fold}/10 | OUTER | "
            f"amostras={len(X_outer_all_pca)} | GPU"
        )
        started_at = time.perf_counter()
        Xq_outer_values = HEISENBERG_PQFM_19Q.transform(X_outer_all_pca)
        q_columns = [
            f"heisenberg_q_{i}" for i in range(Xq_outer_values.shape[1])
        ]
        Xq_outer_all = pd.DataFrame(Xq_outer_values, columns=q_columns)
        print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        quantum_folds.append({
            "classic_fold": classic_fold,
            "inner_quantum": inner_quantum,
            "X_outer_train_quantum": Xq_outer_all.iloc[:n_outer_train].reset_index(drop=True),
            "X_outer_test_quantum": Xq_outer_all.iloc[n_outer_train:].reset_index(drop=True),
        })

    HEISENBERG_PCA_CACHE_19Q_N2000 = {
        "cache_key": HEISENBERG_CACHE_KEY_19Q,
        "config": RUN_LABEL,
        "pca_representation": PCA_REPRESENTATION,
        "n_qubits": N_QUBITS,
        "n_input_features": N_PCA_COMPONENTS,
        "n_blocks": HEISENBERG_PQFM_19Q.theta_info["num_blocks"],
        "n_quantum_features": len(q_columns),
        "folds": quantum_folds,
    }
    print(
        f"Cache concluído: {len(quantum_folds)} outer folds | "
        f"{len(q_columns)} features quânticas por amostra."
    )


Preparando uma única vez o circuito Heisenberg 19q na GPU...
[Heisenberg-19q 01/40] outer=1/10 | inner=1/3 | amostras=96 | GPU
  concluído em 36.7s
[Heisenberg-19q 02/40] outer=1/10 | inner=2/3 | amostras=143 | GPU
  concluído em 55.1s
[Heisenberg-19q 03/40] outer=1/10 | inner=3/3 | amostras=190 | GPU
  concluído em 72.7s
[Heisenberg-19q 04/40] outer=1/10 | OUTER | amostras=371 | GPU
  concluído em 146.4s
[Heisenberg-19q 05/40] outer=2/10 | inner=1/3 | amostras=187 | GPU
  concluído em 76.6s
[Heisenberg-19q 06/40] outer=2/10 | inner=2/3 | amostras=279 | GPU
  concluído em 116.8s
[Heisenberg-19q 07/40] outer=2/10 | inner=3/3 | amostras=371 | GPU
  concluído em 153.8s
[Heisenberg-19q 08/40] outer=2/10 | OUTER | amostras=552 | GPU
  concluído em 231.0s
[Heisenberg-19q 09/40] outer=3/10 | inner=1/3 | amostras=276 | GPU
  concluído em 129.2s
[Heisenberg-19q 10/40] outer=3/10 | inner=2/3 | amostras=414 | GPU
  concluído em 183.0s
[Heisenberg-19q 11/40] outer=3/10 | inner=3/3 | amostras=552 |

In [17]:
# Nested tuning e avaliação. Esta célula não chama a PQFM.
RUN_LABEL = HEISENBERG_PCA_CACHE_19Q_N2000["config"]
PCA_REPRESENTATION = HEISENBERG_PCA_CACHE_19Q_N2000["pca_representation"]
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []

for quantum_fold in HEISENBERG_PCA_CACHE_19Q_N2000["folds"]:
    classic_fold = quantum_fold["classic_fold"]
    outer_fold = classic_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] tuning/classificação outer fold {outer_fold}/10")
    scores = {
        "quantum_only": {candidate_key(params): [] for params in candidates},
        "hibrida": {candidate_key(params): [] for params in candidates},
    }

    for inner_data, inner_quantum in zip(
        classic_fold["inner_folds"],
        quantum_fold["inner_quantum"],
        strict=True,
    ):
        X_train_classical = inner_data["train_representations"]["all_features"]
        X_validation_classical = inner_data["validation_representations"]["all_features"]
        X_train_quantum = inner_quantum["X_train_quantum"]
        X_validation_quantum = inner_quantum["X_validation_quantum"]
        X_train_hybrid = pd.concat(
            [X_train_classical.reset_index(drop=True), X_train_quantum], axis=1
        )
        X_validation_hybrid = pd.concat(
            [X_validation_classical.reset_index(drop=True), X_validation_quantum], axis=1
        )

        for params in candidates:
            key = candidate_key(params)
            scores["quantum_only"][key].append(
                fit_and_score_classifier(
                    X_train_quantum,
                    inner_data["y_train"],
                    X_validation_quantum,
                    inner_data["y_validation"],
                    params,
                )
            )
            scores["hibrida"][key].append(
                fit_and_score_classifier(
                    X_train_hybrid,
                    inner_data["y_train"],
                    X_validation_hybrid,
                    inner_data["y_validation"],
                    params,
                )
            )

    best_params = {}
    for scenario in ("quantum_only", "hibrida"):
        mean_scores = {
            key: float(np.mean(values))
            for key, values in scores[scenario].items()
        }
        best_key = max(mean_scores, key=mean_scores.get)
        best_params[scenario] = json.loads(best_key)
        for key, values in scores[scenario].items():
            tuning_rows.append({
                "config": RUN_LABEL,
                "outer_fold": outer_fold,
                "scenario": scenario,
                "params": key,
                "mean_inner_average_precision": np.mean(values),
                "std_inner_average_precision": np.std(values, ddof=1),
            })

    fold_rows.append({
        "config": RUN_LABEL,
        "scenario": "classica",
        "outer_fold": outer_fold,
        "n_train": len(classic_fold["train_idx"]),
        "n_test": len(classic_fold["test_idx"]),
        "test_start_position": int(classic_fold["test_idx"][0]),
        "test_end_position": int(classic_fold["test_idx"][-1]),
        "best_params": json.dumps(
            classic_fold["best_params"]["all_features"], sort_keys=True
        ),
        **classic_fold["classic_metrics"]["all_features"],
    })

    X_outer_train_classical = classic_fold["outer_train_representations"]["all_features"]
    X_outer_test_classical = classic_fold["outer_test_representations"]["all_features"]
    X_outer_train_quantum = quantum_fold["X_outer_train_quantum"]
    X_outer_test_quantum = quantum_fold["X_outer_test_quantum"]
    X_outer_train_hybrid = pd.concat(
        [X_outer_train_classical.reset_index(drop=True), X_outer_train_quantum], axis=1
    )
    X_outer_test_hybrid = pd.concat(
        [X_outer_test_classical.reset_index(drop=True), X_outer_test_quantum], axis=1
    )

    for scenario, X_train_rep, X_test_rep in (
        ("quantum_only", X_outer_train_quantum, X_outer_test_quantum),
        ("hibrida", X_outer_train_hybrid, X_outer_test_hybrid),
    ):
        params = best_params[scenario]
        classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params,
        )
        classifier.fit(X_train_rep, classic_fold["y_outer_train"])
        prediction = classifier.predict(X_test_rep)
        probability = classifier.predict_proba(X_test_rep)[:, 1]
        fold_rows.append({
            "config": RUN_LABEL,
            "scenario": scenario,
            "outer_fold": outer_fold,
            "n_train": len(classic_fold["train_idx"]),
            "n_test": len(classic_fold["test_idx"]),
            "test_start_position": int(classic_fold["test_idx"][0]),
            "test_end_position": int(classic_fold["test_idx"][-1]),
            "best_params": json.dumps(params, sort_keys=True),
            **compute_metrics(
                classic_fold["y_outer_test"], prediction, probability
            ),
        })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        CHECKPOINT_DIR / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl",
        "wb",
    ) as handle:
        pickle.dump(checkpoint, handle)

HEISENBERG_19Q_PCA36_N2000_RESULT = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
print(f"Experimento {RUN_LABEL} concluído.")



[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 1/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 2/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 3/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 4/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 5/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 6/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 7/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 8/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 9/10

[heisenberg_19q_pca36_n2000_gpu] tuning/classificação outer fold 10/10
Experimento heisenberg_19q_pca36_n2000_gpu concluído.


In [18]:
HEISENBERG_19Q_SUMMARY = summarize_quantum_results(
    HEISENBERG_19Q_PCA36_N2000_RESULT["fold_results"]
)
HEISENBERG_19Q_STATS = quantum_statistical_analysis(
    HEISENBERG_19Q_PCA36_N2000_RESULT["fold_results"],
    title="HEISENBERG 19Q × PCA-36 × N2000 — 10 OUTER FOLDS",
)
display(HEISENBERG_19Q_PCA36_N2000_RESULT["fold_results"])
display(HEISENBERG_19Q_SUMMARY)



  HEISENBERG 19Q × PCA-36 × N2000 — 10 OUTER FOLDS
              Comparação     Métrica  Média Base  Média Caso  Δ Médio            IC95% Δ  p-valor                 Status
Clássico vs Quantum-Only     ROC-AUC      0.8437      0.7785  -0.0652 [-0.1053, -0.0314]   0.0098 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbrido     ROC-AUC      0.8437      0.8423  -0.0015  [-0.0152, 0.0126]   0.8457     ↓ Piora (não sig.)
Clássico vs Quantum-Only      PR-AUC      0.9080      0.8781  -0.0300 [-0.0491, -0.0137]   0.0098 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbrido      PR-AUC      0.9080      0.9061  -0.0019  [-0.0097, 0.0057]   0.6953     ↓ Piora (não sig.)
Clássico vs Quantum-Only    Accuracy      0.7707      0.7066  -0.0641 [-0.1144, -0.0221]   0.0156 ⚠️ PIORA SIGNIFICATIVA
     Clássico vs Híbrido    Accuracy      0.7707      0.7801   0.0094  [-0.0061, 0.0249]   0.3164   ↑ Melhora (não sig.)
Clássico vs Quantum-Only   Precision      0.8308      0.7924  -0.0384 [-0.0698, -0.0112]   0.0840    

,config,scenario,outer_fold,n_train,n_test,test_start_position,test_end_position,best_params,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,Specificity
0,heisenberg_19q_pca36_n2000_gpu,classica,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.980402,0.997854,0.966851,0.981595,0.981595,0.981595,0.833333
1,heisenberg_19q_pca36_n2000_gpu,quantum_only,1,190,181,190,370,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.984663,0.998375,0.955801,0.958580,0.993865,0.975904,0.611111
2,heisenberg_19q_pca36_n2000_gpu,hibrida,1,190,181,190,370,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.982618,0.997982,0.966851,0.993711,0.969325,0.981366,0.944444
3,heisenberg_19q_pca36_n2000_gpu,classica,2,371,181,371,551,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.921175,0.945414,0.861878,0.913043,0.875000,0.893617,0.836066
4,heisenberg_19q_pca36_n2000_gpu,quantum_only,2,371,181,371,551,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.714208,0.855400,0.624309,0.788889,0.591667,0.676190,0.688525
5,heisenberg_19q_pca36_n2000_gpu,hibrida,2,371,181,371,551,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.898087,0.937670,0.823204,0.866667,0.866667,0.866667,0.737705
6,heisenberg_19q_pca36_n2000_gpu,classica,3,552,181,552,732,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.899880,0.970726,0.801105,0.899225,0.834532,0.865672,0.690476
7,heisenberg_19q_pca36_n2000_gpu,quantum_only,3,552,181,552,732,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.910415,0.974681,0.823204,0.914729,0.848921,0.880597,0.738095
8,heisenberg_19q_pca36_n2000_gpu,hibrida,3,552,181,552,732,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est...",0.905447,0.972536,0.812155,0.941176,0.805755,0.868217,0.833333
9,heisenberg_19q_pca36_n2000_gpu,classica,4,733,181,733,913,"{""learning_rate"": 0.05, ""max_depth"": 5, ""n_est...",0.840548,0.937164,0.762431,0.854701,0.793651,0.823045,0.690909


,config,scenario,metric,n_folds,mean,std,ci95_low,ci95_high
0,heisenberg_19q_pca36_n2000_gpu,classica,ROC-AUC,10,0.843733,0.081463,0.785458,0.902008
1,heisenberg_19q_pca36_n2000_gpu,classica,PR-AUC,10,0.908034,0.086030,0.846492,0.969576
2,heisenberg_19q_pca36_n2000_gpu,classica,Accuracy,10,0.770718,0.093079,0.704133,0.837303
3,heisenberg_19q_pca36_n2000_gpu,classica,Precision,10,0.830759,0.090734,0.765852,0.895666
4,heisenberg_19q_pca36_n2000_gpu,classica,Recall,10,0.812332,0.086797,0.750241,0.874423
5,heisenberg_19q_pca36_n2000_gpu,classica,F1,10,0.820493,0.085396,0.759405,0.881582
6,heisenberg_19q_pca36_n2000_gpu,classica,Specificity,10,0.678487,0.088858,0.614921,0.742052
7,heisenberg_19q_pca36_n2000_gpu,quantum_only,ROC-AUC,10,0.778528,0.115332,0.696024,0.861032
8,heisenberg_19q_pca36_n2000_gpu,quantum_only,PR-AUC,10,0.878059,0.094150,0.810708,0.945410
9,heisenberg_19q_pca36_n2000_gpu,quantum_only,Accuracy,10,0.706630,0.123360,0.618384,0.794876


## Parte II.C — Diagnóstico de tempo e bond dimensions: Heisenberg 37q/MPS

Este diagnóstico usa duas linhas do outer test do fold 10. Preprocessing e PCA-36 são ajustados somente no respectivo outer train; as duas linhas recebem apenas `transform`. Primeiro a Heisenberg é executada sem limite de bond dimension e com `mps_log_data=True`. Depois, o mesmo teste é repetido com bond dimension limitada.

O tempo por linha é uma aproximação: com apenas duas linhas, o overhead fixo do Estimator tem peso elevado e o escalonamento para lotes maiores pode não ser perfeitamente linear. A projeção para 1.000 amostras estima apenas custo computacional: PCA-36 continua metodologicamente inviável no primeiro inner train dessa amostra, que permite no máximo 24 componentes centralizados.


In [12]:
# ── DUAS LINHAS PARA O BENCHMARK MPS ────────────────────────────────
TIMING_OUTER_FOLD = 10
TIMING_N_ROWS = 2

timing_train_idx, timing_test_idx = OUTER_FOLDS_2000[TIMING_OUTER_FOLD - 1]
X_timing_train_raw = X_2000.iloc[timing_train_idx].copy()
y_timing_train = y_2000.iloc[timing_train_idx].reset_index(drop=True)
X_timing_two_raw = X_2000.iloc[timing_test_idx[:TIMING_N_ROWS]].copy()

# Ajuste somente no outer train; as duas linhas de teste recebem apenas transform.
TIMING_PREPROCESSING_37Q, X_timing_train_all = fit_preprocessing(
    X_timing_train_raw
)
X_TIMING_TWO_ALL = transform_preprocessing(
    TIMING_PREPROCESSING_37Q,
    X_timing_two_raw,
)
TIMING_PCA_37Q, _X_timing_train_pca36 = fit_pca_reducer(
    X_timing_train_all,
    n_components=36,
)
X_TIMING_TWO_PCA36 = transform_pca(
    TIMING_PCA_37Q,
    X_TIMING_TWO_ALL,
)

# Confirma que a reconstrução coincide com o cache leakage-safe do benchmark.
X_timing_cached = (
    CLASSICAL_BENCHMARK_N2000["folds"][TIMING_OUTER_FOLD - 1]
    ["outer_test_representations"]["pca36"]
    .iloc[:TIMING_N_ROWS]
    .reset_index(drop=True)
)
assert np.allclose(X_TIMING_TWO_PCA36, X_timing_cached)
assert X_TIMING_TWO_PCA36.shape == (TIMING_N_ROWS, 36)

# Quantidade real de linhas processadas nos 30 inner + 10 outer transforms.
# O cálculo depende somente dos tamanhos amostrais, não da execução da Parte I.
MPS_WORKLOAD_ROWS = {}
for workload_label, workload_n_samples in (
    ("n1000_workload_real", 1_000),
    ("n2000_workload_real", 2_000),
):
    row_transforms = 0
    transform_calls = 0
    workload_outer_cv = TimeSeriesSplit(n_splits=OUTER_SPLITS)
    for workload_train_idx, workload_test_idx in workload_outer_cv.split(
        np.arange(workload_n_samples)
    ):
        workload_inner_cv = TimeSeriesSplit(n_splits=INNER_SPLITS)
        for workload_inner_train, workload_inner_validation in (
            workload_inner_cv.split(workload_train_idx)
        ):
            row_transforms += len(workload_inner_train)
            row_transforms += len(workload_inner_validation)
            transform_calls += 1
        row_transforms += len(workload_train_idx)
        row_transforms += len(workload_test_idx)
        transform_calls += 1
    MPS_WORKLOAD_ROWS[workload_label] = {
        "transform_calls": transform_calls,
        "row_transforms": row_transforms,
    }

timing_pipeline_summary = pd.DataFrame([
    {"stage": "raw", "rows": len(X_timing_two_raw), "features": X_timing_two_raw.shape[1]},
    {"stage": "preprocessing", "rows": len(X_TIMING_TWO_ALL), "features": X_TIMING_TWO_ALL.shape[1]},
    {"stage": "PCA-36", "rows": len(X_TIMING_TWO_PCA36), "features": X_TIMING_TWO_PCA36.shape[1]},
])
display(timing_pipeline_summary)
display(pd.DataFrame(MPS_WORKLOAD_ROWS).T)
display(X_TIMING_TWO_PCA36)


,stage,rows,features
0,raw,2,36
1,preprocessing,2,78
2,PCA-36,2,36


,transform_calls,row_transforms
n1000_workload_real,40,17320
n2000_workload_real,40,34469


,PC001,PC002,PC003,PC004,PC005,PC006,PC007,PC008,PC009,PC010,...,PC027,PC028,PC029,PC030,PC031,PC032,PC033,PC034,PC035,PC036
0,-2.666368,2.143983,-0.946535,3.510733,0.149419,-1.228328,1.747141,-0.871413,-1.781509,0.343972,...,0.241321,-1.315711,-0.934489,1.424819,-1.834753,-0.238014,0.388457,-1.018324,0.820946,1.834358
1,1.858655,5.081414,4.553847,0.950828,-2.579757,0.248203,2.014179,0.743017,-1.970098,-2.074839,...,0.814474,-1.487999,0.513256,1.303241,-0.626382,-0.126661,-0.087956,-0.751704,0.379292,-0.012970


In [13]:
# ── TESTE 1: MPS SEM LIMITE DE BOND DIMENSION ──────────────────────
TIMING_RUN_LABEL_UNRESTRICTED = "heisenberg_37q_pca36_timing_unrestricted_mps"
TIMING_SHOTS = SHOTS

HEISENBERG_TIMING_37Q_UNRESTRICTED = HeisenbergProjectiveQFM(
    name_file=TIMING_RUN_LABEL_UNRESTRICTED,
    seed=RANDOM_STATE,
    ideal=True,
    simulation=True,
    fakebackend=False,
    shots=TIMING_SHOTS,
    q_enc=37,
    R=2,
    alpha=0.1,
    use_tanh_scaling=True,
    measure_2local_diagonal=False,
    mps=True,
    use_gpu_statevector=False,
    mps_max_bond_dimension=None,
    mps_truncation_threshold=1e-16,
    output_root=f"outputs/pqfm/{TIMING_RUN_LABEL_UNRESTRICTED}",
)
HEISENBERG_TIMING_37Q_UNRESTRICTED.fit(
    np.zeros((1, 36), dtype=float)
)
HEISENBERG_TIMING_37Q_UNRESTRICTED.backend.set_options(mps_log_data=True)

# O transformer retorna as features, mas não o metadata interno do Aer.
# Esta captura temporária preserva os AerJobs somente para ler MPS_log_data.
MPS_JOBS_UNRESTRICTED = []
_mps_original_run_unrestricted = (
    HEISENBERG_TIMING_37Q_UNRESTRICTED.backend.run
)
HEISENBERG_TIMING_37Q_UNRESTRICTED.backend.run = (
    lambda *args, **kwargs:
        MPS_JOBS_UNRESTRICTED.append(
            _mps_original_run_unrestricted(*args, **kwargs)
        )
        or MPS_JOBS_UNRESTRICTED[-1]
)

try:
    _mps_started_unrestricted = time.perf_counter()
    XQ_TIMING_UNRESTRICTED_37Q = (
        HEISENBERG_TIMING_37Q_UNRESTRICTED.transform(
            X_TIMING_TWO_PCA36
        )
    )
    MPS_SECONDS_UNRESTRICTED = (
        time.perf_counter() - _mps_started_unrestricted
    )
finally:
    HEISENBERG_TIMING_37Q_UNRESTRICTED.backend.run = (
        _mps_original_run_unrestricted
    )

MPS_BOND_DIMENSIONS_UNRESTRICTED = []
MPS_AER_ROWS_UNRESTRICTED = []
for _job_number, _aer_job in enumerate(MPS_JOBS_UNRESTRICTED, 1):
    for _experiment_number, _experiment in enumerate(
        _aer_job.result().results, 1
    ):
        _metadata = _experiment.metadata
        _mps_log = _metadata.get("MPS_log_data", "")
        for _bond_block in re.findall(r"BD=\[([^]]+)\]", _mps_log):
            MPS_BOND_DIMENSIONS_UNRESTRICTED.extend(
                int(value) for value in _bond_block.split()
            )
        MPS_AER_ROWS_UNRESTRICTED.append({
            "job": _job_number,
            "experiment": _experiment_number,
            "time_taken_s": _metadata.get("time_taken", np.nan),
            "sample_measure_time_s": _metadata.get("sample_measure_time", np.nan),
            "method": _metadata.get("method"),
            "device": _metadata.get("device"),
        })

if not MPS_BOND_DIMENSIONS_UNRESTRICTED:
    raise RuntimeError("O Aer não devolveu bond dimensions em MPS_log_data.")

MPS_MAX_BOND_UNRESTRICTED = max(MPS_BOND_DIMENSIONS_UNRESTRICTED)
MPS_SECONDS_PER_ROW_UNRESTRICTED = (
    MPS_SECONDS_UNRESTRICTED / TIMING_N_ROWS
)

_projection_rows_unrestricted = [
    {"workload": "40 × 1.000 (conservador)", "row_transforms": 40_000},
    {"workload": "40 × 2.000 (conservador)", "row_transforms": 80_000},
]
for _label, _workload in MPS_WORKLOAD_ROWS.items():
    _projection_rows_unrestricted.append({
        "workload": _label,
        "row_transforms": _workload["row_transforms"],
    })
for _row in _projection_rows_unrestricted:
    _row["estimated_hours"] = (
        _row["row_transforms"]
        * MPS_SECONDS_PER_ROW_UNRESTRICTED
        / 3600
    )

MPS_TIMING_UNRESTRICTED_SUMMARY = pd.DataFrame([{
    "mode": "unrestricted",
    "rows": TIMING_N_ROWS,
    "quantum_features": XQ_TIMING_UNRESTRICTED_37Q.shape[1],
    "seconds_total": MPS_SECONDS_UNRESTRICTED,
    "seconds_per_row": MPS_SECONDS_PER_ROW_UNRESTRICTED,
    "max_bond_observed": MPS_MAX_BOND_UNRESTRICTED,
    "bond_limit": None,
    "truncation_threshold": 1e-16,
}])
MPS_BOND_COUNTS_UNRESTRICTED = (
    pd.Series(MPS_BOND_DIMENSIONS_UNRESTRICTED, name="bond_dimension")
    .value_counts()
    .sort_index()
    .rename_axis("bond_dimension")
    .reset_index(name="occurrences")
)

display(MPS_TIMING_UNRESTRICTED_SUMMARY)
display(MPS_BOND_COUNTS_UNRESTRICTED)
display(pd.DataFrame(_projection_rows_unrestricted))
display(pd.DataFrame(XQ_TIMING_UNRESTRICTED_37Q))


,mode,rows,quantum_features,seconds_total,seconds_per_row,max_bond_observed,bond_limit,truncation_threshold
0,unrestricted,2,111,3.44361,1.721805,10,None,1.000000e-16


,bond_dimension,occurrences
0,1,39690
1,2,45780
2,3,36
3,4,42501
4,5,2865
5,6,4101
6,7,11448
7,8,16083
8,9,516
9,10,276


,workload,row_transforms,estimated_hours
0,40 × 1.000 (conservador),40000,19.131167
1,40 × 2.000 (conservador),80000,38.262335
2,n1000_workload_real,17320,8.283795
3,n2000_workload_real,34469,16.485805


,0,1,2,3,4,5,6,7,8,9,...,101,102,103,104,105,106,107,108,109,110
0,0.660645,0.473633,0.187500,-0.771484,0.139160,-0.224609,0.152832,0.302246,-0.937988,0.517578,...,-0.945801,0.833496,-0.248535,-0.435059,0.797852,-0.465332,-0.301758,0.784180,0.343262,0.410645
1,0.749512,0.519043,0.446777,-0.838867,0.004395,-0.537598,0.070801,0.357910,-0.936523,0.526367,...,-0.947266,0.722656,-0.428223,-0.404785,0.836914,0.247559,-0.409180,0.743164,-0.342773,0.550293


In [15]:
# ── TESTE 2: MPS COM BOND DIMENSION LIMITADA ───────────────────────
# Valor inicial: metade da maior bond observada, nunca maior que 64.
# Edite esta variável se quiser testar outro limite.
MPS_LIMITED_MAX_BOND_DIMENSION = min(
    64,
    max(1, MPS_MAX_BOND_UNRESTRICTED // 2),
)
TIMING_RUN_LABEL_LIMITED = (
    f"heisenberg_37q_pca36_timing_bond_{MPS_LIMITED_MAX_BOND_DIMENSION}"
)

HEISENBERG_TIMING_37Q_LIMITED = HeisenbergProjectiveQFM(
    name_file=TIMING_RUN_LABEL_LIMITED,
    seed=RANDOM_STATE,
    ideal=True,
    simulation=True,
    fakebackend=False,
    shots=TIMING_SHOTS,
    q_enc=37,
    R=2,
    alpha=0.1,
    use_tanh_scaling=True,
    measure_2local_diagonal=False,
    mps=True,
    use_gpu_statevector=False,
    mps_max_bond_dimension=MPS_LIMITED_MAX_BOND_DIMENSION,
    mps_truncation_threshold=1e-16,
    output_root=f"outputs/pqfm/{TIMING_RUN_LABEL_LIMITED}",
)
HEISENBERG_TIMING_37Q_LIMITED.fit(
    np.zeros((1, 36), dtype=float)
)
HEISENBERG_TIMING_37Q_LIMITED.backend.set_options(mps_log_data=True)

MPS_JOBS_LIMITED = []
_mps_original_run_limited = HEISENBERG_TIMING_37Q_LIMITED.backend.run
HEISENBERG_TIMING_37Q_LIMITED.backend.run = (
    lambda *args, **kwargs:
        MPS_JOBS_LIMITED.append(
            _mps_original_run_limited(*args, **kwargs)
        )
        or MPS_JOBS_LIMITED[-1]
)

try:
    _mps_started_limited = time.perf_counter()
    XQ_TIMING_LIMITED_37Q = HEISENBERG_TIMING_37Q_LIMITED.transform(
        X_TIMING_TWO_PCA36
    )
    MPS_SECONDS_LIMITED = time.perf_counter() - _mps_started_limited
finally:
    HEISENBERG_TIMING_37Q_LIMITED.backend.run = (
        _mps_original_run_limited
    )

MPS_BOND_DIMENSIONS_LIMITED = []
MPS_AER_ROWS_LIMITED = []
for _job_number, _aer_job in enumerate(MPS_JOBS_LIMITED, 1):
    for _experiment_number, _experiment in enumerate(
        _aer_job.result().results, 1
    ):
        _metadata = _experiment.metadata
        _mps_log = _metadata.get("MPS_log_data", "")
        for _bond_block in re.findall(r"BD=\[([^]]+)\]", _mps_log):
            MPS_BOND_DIMENSIONS_LIMITED.extend(
                int(value) for value in _bond_block.split()
            )
        MPS_AER_ROWS_LIMITED.append({
            "job": _job_number,
            "experiment": _experiment_number,
            "time_taken_s": _metadata.get("time_taken", np.nan),
            "sample_measure_time_s": _metadata.get("sample_measure_time", np.nan),
            "method": _metadata.get("method"),
            "device": _metadata.get("device"),
        })

if not MPS_BOND_DIMENSIONS_LIMITED:
    raise RuntimeError("O Aer não devolveu bond dimensions no teste limitado.")

MPS_SECONDS_PER_ROW_LIMITED = MPS_SECONDS_LIMITED / TIMING_N_ROWS
MPS_ABSOLUTE_DIFFERENCE = np.abs(
    XQ_TIMING_LIMITED_37Q - XQ_TIMING_UNRESTRICTED_37Q
)

_projection_rows_limited = []
for _row in _projection_rows_unrestricted:
    _projection_rows_limited.append({
        "workload": _row["workload"],
        "row_transforms": _row["row_transforms"],
        "estimated_hours_unrestricted": _row["estimated_hours"],
        "estimated_hours_limited": (
            _row["row_transforms"]
            * MPS_SECONDS_PER_ROW_LIMITED
            / 3600
        ),
    })

MPS_TIMING_COMPARISON = pd.DataFrame([
    {
        "mode": "unrestricted",
        "bond_limit": None,
        "max_bond_observed": MPS_MAX_BOND_UNRESTRICTED,
        "seconds_total": MPS_SECONDS_UNRESTRICTED,
        "seconds_per_row": MPS_SECONDS_PER_ROW_UNRESTRICTED,
        "speedup_vs_unrestricted": 1.0,
        "mean_abs_feature_difference": 0.0,
        "max_abs_feature_difference": 0.0,
    },
    {
        "mode": "limited",
        "bond_limit": MPS_LIMITED_MAX_BOND_DIMENSION,
        "max_bond_observed": max(MPS_BOND_DIMENSIONS_LIMITED),
        "seconds_total": MPS_SECONDS_LIMITED,
        "seconds_per_row": MPS_SECONDS_PER_ROW_LIMITED,
        "speedup_vs_unrestricted": (
            MPS_SECONDS_UNRESTRICTED / MPS_SECONDS_LIMITED
        ),
        "mean_abs_feature_difference": MPS_ABSOLUTE_DIFFERENCE.mean(),
        "max_abs_feature_difference": MPS_ABSOLUTE_DIFFERENCE.max(),
    },
])
MPS_BOND_COUNTS_LIMITED = (
    pd.Series(MPS_BOND_DIMENSIONS_LIMITED, name="bond_dimension")
    .value_counts()
    .sort_index()
    .rename_axis("bond_dimension")
    .reset_index(name="occurrences")
)

display(MPS_TIMING_COMPARISON)
display(MPS_BOND_COUNTS_LIMITED)
display(pd.DataFrame(_projection_rows_limited))
display(pd.DataFrame(XQ_TIMING_LIMITED_37Q))


,mode,bond_limit,max_bond_observed,seconds_total,seconds_per_row,speedup_vs_unrestricted,mean_abs_feature_difference,max_abs_feature_difference
0,unrestricted,NaN,10,3.443610,1.721805,1.000000,0.0,0.0
1,limited,5.0,10,3.288037,1.644018,1.047315,0.0,0.0


,bond_dimension,occurrences
0,1,107730
1,2,124260
2,3,90
3,4,115383
4,5,39159
5,6,7200
6,7,19386
7,8,28566
8,9,630
9,10,828


,workload,row_transforms,estimated_hours_unrestricted,estimated_hours_limited
0,40 × 1.000 (conservador),40000,19.131167,18.266870
1,40 × 2.000 (conservador),80000,38.262335,36.533740
2,n1000_workload_real,17320,8.283795,7.909555
3,n2000_workload_real,34469,16.485805,15.741018


,0,1,2,3,4,5,6,7,8,9,...,101,102,103,104,105,106,107,108,109,110
0,0.660645,0.473633,0.187500,-0.771484,0.139160,-0.224609,0.152832,0.302246,-0.937988,0.517578,...,-0.945801,0.833496,-0.248535,-0.435059,0.797852,-0.465332,-0.301758,0.784180,0.343262,0.410645
1,0.749512,0.519043,0.446777,-0.838867,0.004395,-0.537598,0.070801,0.357910,-0.936523,0.526367,...,-0.947266,0.722656,-0.428223,-0.404785,0.836914,0.247559,-0.409180,0.743164,-0.342773,0.550293


## Parte II.D — Heisenberg 37q × PCA-36, CPU/MPS

Trinta e sete qubits formam uma cadeia com 36 arestas. As 36 componentes PCA ocupam exatamente um bloco. O backend usa `matrix_product_state` em CPU, sem limite de bond dimension e com o limiar conservador padrão `1e-16`.

PCA-36 é usado somente para o encoding quântico. O baseline clássico usa todas as features preprocessadas e o híbrido concatena todas as features clássicas com as 111 features quânticas locais Z/X/Y.


In [ ]:
# ── CONFIGURAÇÃO EDITÁVEL DESTA CÉLULA ──────────────────────────────
RUN_LABEL = "heisenberg_37q_pca36_n2000_cpu_mps"
N_QUBITS = 37
N_PCA_COMPONENTS = 36
HEISENBERG_R = 2
HEISENBERG_ALPHA = 0.1
HEISENBERG_SHOTS = SHOTS
USE_TANH_SCALING = True
MEASURE_2LOCAL_DIAGONAL = False
CLASSICAL_BENCHMARK = CLASSICAL_BENCHMARK_N2000
PCA_REPRESENTATION = "pca36"
MPS_MAX_BOND_DIMENSION = None
MPS_TRUNCATION_THRESHOLD = 1e-16
# ────────────────────────────────────────────────────────────────────

HEISENBERG_CACHE_KEY_37Q = (
    RUN_LABEL,
    id(CLASSICAL_BENCHMARK),
    RANDOM_STATE,
    N_QUBITS,
    N_PCA_COMPONENTS,
    HEISENBERG_R,
    HEISENBERG_ALPHA,
    HEISENBERG_SHOTS,
    USE_TANH_SCALING,
    MEASURE_2LOCAL_DIAGONAL,
    MPS_MAX_BOND_DIMENSION,
    MPS_TRUNCATION_THRESHOLD,
)
cache_is_current = (
    "HEISENBERG_PCA_CACHE_37Q_N2000" in globals()
    and HEISENBERG_PCA_CACHE_37Q_N2000.get("cache_key")
    == HEISENBERG_CACHE_KEY_37Q
)

if cache_is_current:
    print("Cache Heisenberg 37q/PCA-36/n2000 CPU-MPS já existe; reutilizando.")
else:
    HEISENBERG_PQFM_37Q = HeisenbergProjectiveQFM(
        name_file=RUN_LABEL,
        seed=RANDOM_STATE,
        ideal=True,
        simulation=True,
        fakebackend=False,
        shots=HEISENBERG_SHOTS,
        q_enc=N_QUBITS,
        R=HEISENBERG_R,
        alpha=HEISENBERG_ALPHA,
        use_tanh_scaling=USE_TANH_SCALING,
        measure_2local_diagonal=MEASURE_2LOCAL_DIAGONAL,
        mps=True,
        use_gpu_statevector=False,
        mps_max_bond_dimension=MPS_MAX_BOND_DIMENSION,
        mps_truncation_threshold=MPS_TRUNCATION_THRESHOLD,
        output_root=f"outputs/pqfm/{RUN_LABEL}",
    )
    print("Preparando uma única vez o circuito Heisenberg 37q em CPU/MPS...")
    HEISENBERG_PQFM_37Q.fit(
        np.zeros((1, N_PCA_COMPONENTS), dtype=float)
    )
    assert HEISENBERG_PQFM_37Q.backend.options.method == "matrix_product_state"
    assert HEISENBERG_PQFM_37Q.backend.options.device == "CPU"
    assert HEISENBERG_PQFM_37Q.theta_info["features_per_block"] == 36
    assert HEISENBERG_PQFM_37Q.theta_info["num_blocks"] == 1
    assert HEISENBERG_PQFM_37Q.theta_info["total_slots"] == 36

    quantum_folds = []
    transform_counter = 0
    transform_total = len(CLASSICAL_BENCHMARK["folds"]) * (INNER_SPLITS + 1)

    for classic_fold in CLASSICAL_BENCHMARK["folds"]:
        outer_fold = classic_fold["outer_fold"]
        inner_quantum = []
        for inner_data in classic_fold["inner_folds"]:
            X_train_pca = inner_data["train_representations"][PCA_REPRESENTATION]
            X_validation_pca = inner_data["validation_representations"][PCA_REPRESENTATION]
            n_train = len(X_train_pca)
            X_all_pca = pd.concat(
                [X_train_pca, X_validation_pca],
                ignore_index=True,
            )
            transform_counter += 1
            print(
                f"[Heisenberg-37q {transform_counter:02d}/{transform_total:02d}] "
                f"outer={outer_fold}/10 | inner={inner_data['inner_fold']}/3 | "
                f"amostras={len(X_all_pca)} | CPU/MPS"
            )
            started_at = time.perf_counter()
            Xq_values = HEISENBERG_PQFM_37Q.transform(X_all_pca)
            q_columns = [f"heisenberg_q_{i}" for i in range(Xq_values.shape[1])]
            Xq_all = pd.DataFrame(Xq_values, columns=q_columns)
            inner_quantum.append({
                "inner_fold": inner_data["inner_fold"],
                "X_train_quantum": Xq_all.iloc[:n_train].reset_index(drop=True),
                "X_validation_quantum": Xq_all.iloc[n_train:].reset_index(drop=True),
            })
            print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        X_outer_train_pca = classic_fold["outer_train_representations"][PCA_REPRESENTATION]
        X_outer_test_pca = classic_fold["outer_test_representations"][PCA_REPRESENTATION]
        n_outer_train = len(X_outer_train_pca)
        X_outer_all_pca = pd.concat(
            [X_outer_train_pca, X_outer_test_pca],
            ignore_index=True,
        )
        transform_counter += 1
        print(
            f"[Heisenberg-37q {transform_counter:02d}/{transform_total:02d}] "
            f"outer={outer_fold}/10 | OUTER | "
            f"amostras={len(X_outer_all_pca)} | CPU/MPS"
        )
        started_at = time.perf_counter()
        Xq_outer_values = HEISENBERG_PQFM_37Q.transform(X_outer_all_pca)
        q_columns = [
            f"heisenberg_q_{i}" for i in range(Xq_outer_values.shape[1])
        ]
        Xq_outer_all = pd.DataFrame(Xq_outer_values, columns=q_columns)
        print(f"  concluído em {time.perf_counter() - started_at:.1f}s")

        quantum_folds.append({
            "classic_fold": classic_fold,
            "inner_quantum": inner_quantum,
            "X_outer_train_quantum": Xq_outer_all.iloc[:n_outer_train].reset_index(drop=True),
            "X_outer_test_quantum": Xq_outer_all.iloc[n_outer_train:].reset_index(drop=True),
        })

    HEISENBERG_PCA_CACHE_37Q_N2000 = {
        "cache_key": HEISENBERG_CACHE_KEY_37Q,
        "config": RUN_LABEL,
        "pca_representation": PCA_REPRESENTATION,
        "n_qubits": N_QUBITS,
        "n_input_features": N_PCA_COMPONENTS,
        "n_blocks": HEISENBERG_PQFM_37Q.theta_info["num_blocks"],
        "n_quantum_features": len(q_columns),
        "folds": quantum_folds,
    }
    print(
        f"Cache concluído: {len(quantum_folds)} outer folds | "
        f"{len(q_columns)} features quânticas por amostra."
    )


Preparando uma única vez o circuito Heisenberg 37q em CPU/MPS...
[Heisenberg-37q 01/40] outer=1/10 | inner=1/3 | amostras=96 | CPU/MPS
  concluído em 192.7s
[Heisenberg-37q 02/40] outer=1/10 | inner=2/3 | amostras=143 | CPU/MPS
  concluído em 291.5s
[Heisenberg-37q 03/40] outer=1/10 | inner=3/3 | amostras=190 | CPU/MPS
  concluído em 391.9s
[Heisenberg-37q 04/40] outer=1/10 | OUTER | amostras=371 | CPU/MPS


In [ ]:
# Nested tuning e avaliação. Esta célula não chama a PQFM.
RUN_LABEL = HEISENBERG_PCA_CACHE_37Q_N2000["config"]
candidates = list(ParameterGrid(CLASSIFIER_GRID))
fold_rows = []
tuning_rows = []

for quantum_fold in HEISENBERG_PCA_CACHE_37Q_N2000["folds"]:
    classic_fold = quantum_fold["classic_fold"]
    outer_fold = classic_fold["outer_fold"]
    print(f"\n[{RUN_LABEL}] tuning/classificação outer fold {outer_fold}/10")
    scores = {
        "quantum_only": {candidate_key(params): [] for params in candidates},
        "hibrida": {candidate_key(params): [] for params in candidates},
    }

    for inner_data, inner_quantum in zip(
        classic_fold["inner_folds"],
        quantum_fold["inner_quantum"],
        strict=True,
    ):
        X_train_classical = inner_data["train_representations"]["all_features"]
        X_validation_classical = inner_data["validation_representations"]["all_features"]
        X_train_quantum = inner_quantum["X_train_quantum"]
        X_validation_quantum = inner_quantum["X_validation_quantum"]
        X_train_hybrid = pd.concat(
            [X_train_classical.reset_index(drop=True), X_train_quantum], axis=1
        )
        X_validation_hybrid = pd.concat(
            [X_validation_classical.reset_index(drop=True), X_validation_quantum], axis=1
        )

        for params in candidates:
            key = candidate_key(params)
            scores["quantum_only"][key].append(
                fit_and_score_classifier(
                    X_train_quantum,
                    inner_data["y_train"],
                    X_validation_quantum,
                    inner_data["y_validation"],
                    params,
                )
            )
            scores["hibrida"][key].append(
                fit_and_score_classifier(
                    X_train_hybrid,
                    inner_data["y_train"],
                    X_validation_hybrid,
                    inner_data["y_validation"],
                    params,
                )
            )

    best_params = {}
    for scenario in ("quantum_only", "hibrida"):
        mean_scores = {
            key: float(np.mean(values))
            for key, values in scores[scenario].items()
        }
        best_key = max(mean_scores, key=mean_scores.get)
        best_params[scenario] = json.loads(best_key)
        for key, values in scores[scenario].items():
            tuning_rows.append({
                "config": RUN_LABEL,
                "outer_fold": outer_fold,
                "scenario": scenario,
                "params": key,
                "mean_inner_average_precision": np.mean(values),
                "std_inner_average_precision": np.std(values, ddof=1),
            })

    fold_rows.append({
        "config": RUN_LABEL,
        "scenario": "classica",
        "outer_fold": outer_fold,
        "n_train": len(classic_fold["train_idx"]),
        "n_test": len(classic_fold["test_idx"]),
        "test_start_position": int(classic_fold["test_idx"][0]),
        "test_end_position": int(classic_fold["test_idx"][-1]),
        "best_params": json.dumps(
            classic_fold["best_params"]["all_features"], sort_keys=True
        ),
        **classic_fold["classic_metrics"]["all_features"],
    })

    X_outer_train_classical = classic_fold["outer_train_representations"]["all_features"]
    X_outer_test_classical = classic_fold["outer_test_representations"]["all_features"]
    X_outer_train_quantum = quantum_fold["X_outer_train_quantum"]
    X_outer_test_quantum = quantum_fold["X_outer_test_quantum"]
    X_outer_train_hybrid = pd.concat(
        [X_outer_train_classical.reset_index(drop=True), X_outer_train_quantum], axis=1
    )
    X_outer_test_hybrid = pd.concat(
        [X_outer_test_classical.reset_index(drop=True), X_outer_test_quantum], axis=1
    )

    for scenario, X_train_rep, X_test_rep in (
        ("quantum_only", X_outer_train_quantum, X_outer_test_quantum),
        ("hibrida", X_outer_train_hybrid, X_outer_test_hybrid),
    ):
        params = best_params[scenario]
        classifier = GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params,
        )
        classifier.fit(X_train_rep, classic_fold["y_outer_train"])
        prediction = classifier.predict(X_test_rep)
        probability = classifier.predict_proba(X_test_rep)[:, 1]
        fold_rows.append({
            "config": RUN_LABEL,
            "scenario": scenario,
            "outer_fold": outer_fold,
            "n_train": len(classic_fold["train_idx"]),
            "n_test": len(classic_fold["test_idx"]),
            "test_start_position": int(classic_fold["test_idx"][0]),
            "test_end_position": int(classic_fold["test_idx"][-1]),
            "best_params": json.dumps(params, sort_keys=True),
            **compute_metrics(
                classic_fold["y_outer_test"], prediction, probability
            ),
        })

    checkpoint = {
        "config": RUN_LABEL,
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.DataFrame(tuning_rows),
    }
    with open(
        CHECKPOINT_DIR / f"{RUN_LABEL}_fold_{outer_fold:02d}.pkl",
        "wb",
    ) as handle:
        pickle.dump(checkpoint, handle)

HEISENBERG_37Q_PCA36_N2000_RESULT = {
    "config": RUN_LABEL,
    "fold_results": pd.DataFrame(fold_rows),
    "tuning_results": pd.DataFrame(tuning_rows),
}
print(f"Experimento {RUN_LABEL} concluído.")


In [ ]:
HEISENBERG_37Q_SUMMARY = summarize_quantum_results(
    HEISENBERG_37Q_PCA36_N2000_RESULT["fold_results"]
)
HEISENBERG_37Q_STATS = quantum_statistical_analysis(
    HEISENBERG_37Q_PCA36_N2000_RESULT["fold_results"],
    title="HEISENBERG 37Q × PCA-36 × N2000 × CPU/MPS — 10 OUTER FOLDS",
)
display(HEISENBERG_37Q_PCA36_N2000_RESULT["fold_results"])
display(HEISENBERG_37Q_SUMMARY)


## Resumo descritivo final

As amostras de 1.000 e 2.000 registros são diferentes. Portanto, esta tabela é apenas descritiva; testes pareados são feitos somente dentro de cada tamanho amostral e arquitetura.


In [ ]:
FINAL_CLASSICAL_SUMMARY = pd.concat(
    [CLASSICAL_SUMMARY_N1000, CLASSICAL_SUMMARY_N2000],
    ignore_index=True,
)
FINAL_HEISENBERG_SUMMARY = pd.concat(
    [HEISENBERG_13Q_SUMMARY, HEISENBERG_19Q_SUMMARY, HEISENBERG_37Q_SUMMARY],
    ignore_index=True,
)

display(FINAL_CLASSICAL_SUMMARY)
display(FINAL_HEISENBERG_SUMMARY)
